# Conservative Seasonal Period Detection Experiment

This notebook tests seasonal-period detection on synthetic series generated by `TimeSeriesGenerator`.

Important design choices:

1. **Plots are saved to disk only** by default. They are not shown on screen, so the notebook should not crash from too many figures.
2. The detector does **not** use `info["periods"]`. True periods are used only for evaluation and plotting.
3. Evaluation includes both:
   - `success_any_rate` and `success_all_rate`, and
   - **period-level TP, TN, FP, FN counts/rates**.
4. TP/TN/FP/FN are calculated at the **candidate-period level**. For example, if a series has true periods `[7, 30, 52]` and the detector finds `[7, 30]`, then it gets **2 TP** and **1 FN** for that series. Extra detected periods are counted as **FP**.

The detector is intentionally conservative: it returns `[]` when no period is reliable enough.


In [1]:
# =========================
# Main experiment settings
# =========================

import numpy as np

OUTPUT_DIR = "period_detection_experiment_2"

# Keep this small for debugging. Increase later after the notebook works.
EXAMPLES_PER_TYPE = 25

# Longer series make period detection easier but take longer.
LENGTH = np.random.randint(100, 1000)

RANDOM_SEED = 42

# Plot behavior: save plots, but DO NOT show them on screen.
SAVE_PLOTS = True
SHOW_PLOTS = False

# Candidate-period filtering.
MIN_CYCLES = 6
MAX_DETECTED_PERIODS = 3

# Fast mode: only run expensive STL/Fourier checks on top candidates.
TOP_CANDIDATES_FOR_EXPENSIVE_TESTS = 8

# Detection thresholds. You can tune these later.
PERIODOGRAM_POWER_THRESHOLD = 0.08
STL_STRENGTH_THRESHOLD = 0.30
BIC_IMPROVEMENT_THRESHOLD = 15.0
MIN_PASS_COUNT = 3

# Period matching tolerance for evaluation.
ABSOLUTE_PERIOD_TOLERANCE = 1.0
RELATIVE_PERIOD_TOLERANCE = 0.10

# Detrending before spectral analysis: "constant", "linear", or None.
DETREND_METHOD = "constant"

# Quiet mode suppresses generator warnings/prints inside loops.
QUIET_MODE = True


In [2]:
# =========================
# Imports
# =========================

import os
import io
import ast
import json
import math
import random
import warnings
from pathlib import Path
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")  # prevents notebook display overload
import matplotlib.pyplot as plt

from scipy.signal import detrend as scipy_detrend
from statsmodels.tsa.seasonal import STL
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

OUTPUT_DIR = Path(OUTPUT_DIR)
CSV_DIR = OUTPUT_DIR / "generated_csv"
PLOT_DIR = OUTPUT_DIR / "plots"
SCORE_DIR = OUTPUT_DIR / "candidate_scores"

for directory in [OUTPUT_DIR, CSV_DIR, PLOT_DIR, SCORE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


## Generator class

This cell contains your uploaded `TimeSeriesGenerator` class. If you change the generator again, replace this cell with the new class definition.

In [3]:
import numpy as np
import pandas as pd
import random
from numpy.polynomial import Polynomial
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.tsa.statespace.sarimax import SARIMAX
from arch import arch_model
from statsmodels.tsa.seasonal import STL,MSTL

class TimeSeriesGenerator:
    def __init__(self, length=None):
        self.length = length if length is not None else 400
        self.stationary_base_distributions = ['ar', 'ma', 'arma','white_noise']
        self.seasonal_base_distributions = ['sarma', 'sarima']
        self.volatile_base_distributions = ['arch', 'garch', 'egarch', 'aparch']
        self.stochastic_base_distributions = ['ari', 'ima', 'arima']
        self.characteristics = {'deterministic_trend_linear' : self.generate_deterministic_trend_linear,
        'deterministic_trend_cubic': self.generate_deterministic_trend_cubic,
        'deterministic_trend_quadratic': self.generate_deterministic_trend_quadratic,
        'deterministic_trend_exponential': self.generate_deterministic_trend_exponential,
        'deterministic_trend_damped': self.generate_deterministic_trend_damped,
        'stochastic_trend': self.generate_stochastic_trend,
        'single_seasonality': self.generate_single_seasonality,
        'multiple_seasonality': self.generate_multiple_seasonality,
        'single_point_anomaly' : self.generate_point_anomaly,
        'multiple_point_anomalies': self.generate_point_anomalies,
        'collective_anomalies': self.generate_collective_anomalies,
        'contextual_anomalies': self.generate_contextual_anomalies}
        self.structural_breaks = {'mean_shift': self.generate_mean_shift,
        'variance_shift': self.generate_variance_shift,
        'trend_shift': self.generate_trend_shift}

    #HELPER FUNCTIONS

    # Check if AR parameters lead to stationarity
    def is_stationary(self, ar_params):
        ar_poly = np.r_[1, -ar_params]
        roots = Polynomial(ar_poly).roots()
        return np.all(np.abs(roots) > 1)

    # Check if MA parameters lead to invertibility
    def is_invertible(self, ma_params):
        ma_poly = np.r_[1, ma_params]
        roots = Polynomial(ma_poly).roots()
        return np.all(np.abs(roots) > 1)

    def generate_nonzero_coefs(self, order, low, high, exclusion_lower, exclusion_upper):
        coefs = []
        while len(coefs) < order:
            val = np.random.uniform(low, high)
            if abs(val) >= exclusion_lower and abs(val) <= exclusion_upper:
                coefs.append(val)
        return np.array(coefs)
        

    #BASE DISTRIBUTIONS STATIONARY

    def generate_ar_params(self, order_range=(1, 5), coef_range=(-0.9, 0.9)):
        while True:
            order = np.random.randint(order_range[0], order_range[1] + 1)
            coefs = np.random.uniform(coef_range[0], coef_range[1], order)
            ar = np.r_[1, -coefs]
            ma = np.array([1])
            arma_process = ArmaProcess(ar, ma)
            if arma_process.isstationary:
                break
        return order, coefs

    def generate_ar_series(self, length, noise_std = None):
        noise_std = noise_std if noise_std is not None else np.random.uniform(0.1, 1.5)
        order,coefs = self.generate_ar_params()
        info = {'type': 'base_series', 'subtype': 'AR', 'ar_order': order, 'ar_coefs': coefs} 
        ar = np.r_[1, -np.array(coefs)]  # leading 1 and negate the coefficients
        ma = np.r_[1]  # MA coefficients are just [1] for a pure AR process
        ar_process = ArmaProcess(ar, ma)
        series = ar_process.generate_sample(nsample=length)
        series = series + np.random.normal(0,noise_std,length)
        return series, info
            
    def generate_ma_params(self, order_range=(1, 5), coef_range=(-0.9, 0.9)):
        while True:
            order = np.random.randint(order_range[0], order_range[1] + 1)
            coefs = np.random.uniform(coef_range[0], coef_range[1], order)
            ma = np.r_[1, coefs]
            ar = np.array([1])
            arma_process = ArmaProcess(ar, ma)
            if arma_process.isinvertible:
                break
        return order, coefs

    def generate_ma_series(self, length, noise_std = None):
        noise_std = noise_std if noise_std is not None else np.random.uniform(0.1, 1.5)
        order,coefs = self.generate_ma_params()
        info = {'type': 'base_series','subtype': 'MA', 'ma_order': order, 'ma_coefs': coefs}
        ar = np.r_[1]  # AR coefficients are just [1] for a pure MA process
        ma = np.r_[1, np.array(coefs)]  # leading 1 for the MA coefficients
        arma_process = ArmaProcess(ar, ma)
        series = arma_process.generate_sample(nsample=length)
        series = series + np.random.normal(0,noise_std,length)
        return series, info

    def generate_arma_params(self, order_range=(1, 5), coef_range=(-0.9, 0.9)):
        while True:
            ar_order = np.random.randint(order_range[0], order_range[1] + 1)
            ma_order = np.random.randint(order_range[0], order_range[1] + 1)
            ar_coefs = np.random.uniform(coef_range[0], coef_range[1], ar_order)
            ma_coefs = np.random.uniform(coef_range[0], coef_range[1], ma_order)
            ma = np.r_[1, ma_coefs]
            ar = np.r_[1, -ar_coefs]
            arma_process = ArmaProcess(ar, ma)
            if arma_process.isinvertible and arma_process.isstationary:
                break
        return ar_order, ma_order, ar_coefs, ma_coefs

    def generate_arma_series(self, length, noise_std = None):
        noise_std = noise_std if noise_std is not None else np.random.uniform(0.1, 1.5)
        ar_order,ma_order,ar_coefs,ma_coefs = self.generate_arma_params()
        info = {'type': 'base_series', 'subtype': 'ARMA', 'ar_order': ar_order, 'ar_coefs': ar_coefs, 'ma_order': ma_order, 'ma_coefs': ma_coefs}
        ar = np.r_[1, -np.array(ar_coefs)]
        ma = np.r_[1, np.array(ma_coefs)]
        arma_process = ArmaProcess(ar, ma)
        series = arma_process.generate_sample(nsample=length)
        series = series + np.random.normal(0,noise_std,length)
        return series, info

    def generate_white_noise(self, length, noise_std = None):
        info = {'type': 'base_series','subtype': 'white_noise'}
        noise_std = noise_std if noise_std is not None else np.random.uniform(0.1, 1.5)
        series = np.random.normal(0, 1, length)
        series = series + np.random.normal(0,noise_std,length)
        return series, info

    def generate_arima_params(self, order_range=(1, 3), coef_range = (-0.9,0.9)):
        while True:
            p = np.random.randint(order_range[0], order_range[1] + 1)
            q = np.random.randint(order_range[0], order_range[1] + 1)

            ar_coefs = self.generate_nonzero_coefs(p, coef_range[0], coef_range[1], exclusion_lower=0.2, exclusion_upper=0.8)
            ma_coefs = self.generate_nonzero_coefs(q, coef_range[0], coef_range[1], exclusion_lower=0.2, exclusion_upper=0.8)

            ar = np.r_[1, -ar_coefs]
            ma = np.r_[1, ma_coefs]

            arma_process = ArmaProcess(ar, ma)
            if arma_process.isstationary and arma_process.isinvertible:
                break

        return p, q, ar_coefs, ma_coefs

    def generate_arima_series(self, length, d= 1, const=False, drift=None, noise_std = None):
        noise_std = noise_std if noise_std is not None else np.random.uniform(0.1, 1.5)
        p, q, ar_coefs, ma_coefs = self.generate_arima_params()

        ar = np.r_[1, -ar_coefs]
        ma = np.r_[1, ma_coefs]

        if d == 1: 
            unit_root_label = "1_unit_root"
        elif d == 2:
            unit_root_label = "2_unit_root"

        info = {'type': 'trend', 'subtype' : 'stochastic_ARIMA', 'unit_root': unit_root_label,
                 'ar_order': p, 'ar_coefs': ar_coefs, 'ma_order': q, 'ma_coefs': ma_coefs, 'diff': d}
        arma_process = ArmaProcess(ar, ma)
        arma_sample = arma_process.generate_sample(nsample=length)

        # Integrate (difference 'd' times)
        series = arma_sample
        for _ in range(d):
            series = np.cumsum(series)
        if const:
            if drift is None:
                drift = np.random.uniform(0.01, 0.08)
            series += drift * np.arange(length)
        series = series + np.random.normal(0,noise_std,length)
        return series, info

    def generate_ari_params(self, order_range=(1, 3), coef_range = (-0.9,0.9)):
        while True:
            order = np.random.randint(order_range[0], order_range[1] + 1)
            coefs = self.generate_nonzero_coefs(order, coef_range[0], coef_range[1], exclusion_lower = 0.3, exclusion_upper = 0.6)
            ar = np.r_[1, -coefs]
            ma = np.array([1])
            arma_process = ArmaProcess(ar, ma)
            if arma_process.isstationary:
                break
        return order, coefs

    def generate_ari_series(self, length, d = 1, const=False, drift=None, noise_std = None):
        noise_std = noise_std if noise_std is not None else np.random.uniform(0.1, 1.5)
        order, coefs = self.generate_ari_params()
        if d == 1: 
            unit_root_label = "1_unit_root"
        elif d == 2:
            unit_root_label = "2_unit_root"
        info = {'type': 'trend', 'subtype' : 'stochastic_ARI', 'unit_root': unit_root_label,
                 'ar_order': order, 'ar_coefs': coefs, 'diff': d}
        ar = np.r_[1, -coefs]
        ma = np.array([1])
        arma_process = ArmaProcess(ar, ma)
        series = arma_process.generate_sample(nsample=length)
        for _ in range(d):
            series = np.cumsum(series)
        if const:
            if drift is None:
                drift = np.random.uniform(0.01, 0.08)
            series += drift * np.arange(length)
        series = series + np.random.normal(0,noise_std,length)
        return series, info

    def generate_ima_params(self, order_range=(1, 3), coef_range = (-0.9,0.9)):
        while True:
            order = np.random.randint(order_range[0], order_range[1] + 1)
            coefs = self.generate_nonzero_coefs(order, coef_range[0], coef_range[1], exclusion_lower = 0.3, exclusion_upper = 0.6)
            ar = np.array([1])
            ma = np.r_[1, coefs]
            arma_process = ArmaProcess(ar, ma)
            if arma_process.isinvertible:
                break
        return order, coefs

    def generate_ima_series(self, length, d = 1, const=False, drift=None, noise_scale=0.5, noise_std = None):
        noise_std = noise_std if noise_std is not None else np.random.uniform(0.1, 1.5)
        order, coefs = self.generate_ima_params()
        if d == 1: 
            unit_root_label = "1_unit_root"
        elif d == 2:
            unit_root_label = "2_unit_root"
        info = {'type': 'trend', 'subtype' : 'stochastic_IMA', 'unit_root': unit_root_label, 
                'ma_order': order, 'ma_coefs': coefs, 'diff': d}
        ar = np.array([1])
        ma = np.r_[1, coefs]
        arma_process = ArmaProcess(ar, ma)
        series = arma_process.generate_sample(nsample=length)
        for _ in range(d):
            series = np.cumsum(series)
        if const:
            if drift is None:
                drift = np.random.uniform(0.01, 0.8)
            series += drift * np.arange(length)
        series = series + np.random.normal(0,noise_std,length)
        return series, info


    
    def generate_arch_series(self, length, alpha_range=(0.5, 0.9), omega_range=(0.1, 0.3), cumulative=False, scale_factor=1):
        alpha = np.random.uniform(*alpha_range)
        omega = np.random.uniform(*omega_range)
        
        am = arch_model(None, vol='ARCH', p=1, mean='Zero')
        sim = am.simulate([omega, alpha], nobs=length)
        
        series = sim['data'].values * scale_factor
        info = {'type': 'volatility', 'subtype': 'ARCH', 'alpha': alpha, 'omega': omega}
        if cumulative:
            series = np.cumsum(series)
    
        return series, info

    def generate_garch_series(self, length, alpha_range=(0.4, 0.6), beta_range=(0.2, 0.5), omega_range=(0.3, 0.6), cumulative=False, scale_factor=1):
        while True:
            alpha = np.random.uniform(*alpha_range)
            beta = np.random.uniform(*beta_range)
            omega = np.random.uniform(*omega_range)
            if alpha + beta < 1:
                break  # Ensure weak stationarity of the variance
    
        am = arch_model(None, vol='GARCH', p=1, q=1, mean='Zero')
        sim = am.simulate([omega, alpha, beta], nobs=length)
        
        series = sim['data'].values * scale_factor
        info = {'type': 'volatility', 'subtype': 'GARCH', 'alpha': alpha, 'beta': beta, 'omega': omega}
        if cumulative:
            series = np.cumsum(series)
    
        return series, info

    def generate_egarch_series(self, length, omega_range=(0.1, 0.3), alpha_range=(0.5, 0.9), beta_range=(0.6, 0.9), theta_range=(-0.5, 0.5), lambda_range=(0.1, 0.5), cumulative=False, scale_factor=1):
        omega = np.random.uniform(*omega_range)
        alpha = np.random.uniform(*alpha_range)
        beta = np.random.uniform(*beta_range)
        theta = np.random.uniform(*theta_range)
        lam = np.random.uniform(*lambda_range)

        am = arch_model(None, vol='EGARCH', p=1, q=1, mean='Zero', dist='normal')
        sim = am.simulate([omega, alpha, beta, theta, lam], nobs=length)

        series = sim['data'].values * scale_factor
        info = {'type': 'volatility', 'subtype': 'EGARCH', 'alpha': alpha, 'beta': beta, 'theta': theta, 'lambda': lam, 'omega': omega}
        if cumulative:
            series = np.cumsum(series)

        return series, info

    def generate_aparch_series(self, length, omega_range=(0.1, 0.3), alpha_range=(0.1, 0.3), beta_range=(0.5, 0.8), gamma_range=(-0.3, 0.3), delta_range=(1.0, 2.0), cumulative=False, scale_factor=1):
        # Stationarity constraint: alpha + beta < 1
        while True:
            alpha = np.random.uniform(*alpha_range)
            beta = np.random.uniform(*beta_range)
            if alpha + beta < 1:
                break
        
        omega = np.random.uniform(*omega_range)
        gamma = np.random.uniform(*gamma_range)
        delta = np.random.uniform(*delta_range)

        from arch import arch_model
        am = arch_model(None, vol='APARCH', p=1, o=1, q=1, mean='Zero', dist='normal')

        sim = am.simulate([omega, alpha, gamma, beta, delta], nobs=length)

        series = sim['data'].values * scale_factor
        info = {'type': 'volatility', 'subtype': 'APARCH', 'alpha': alpha, 'beta': beta, 'gamma': gamma, 'delta': delta, 'omega': omega}
        if cumulative:
            series = np.cumsum(series)

        return series, info

    def generate_stationary_base_series(self, distribution=None):
        if distribution is None:
            distribution = np.random.choice(self.stationary_base_distributions)
        if distribution == 'white_noise':
            series, info = self.generate_white_noise(self.length)
        elif distribution == 'ar':
            series, info = self.generate_ar_series(self.length)
        elif distribution == 'ma':
            series, info = self.generate_ma_series(self.length)
        elif distribution == 'arma':
            series, info = self.generate_arma_series(self.length)

        df = pd.DataFrame({
            'time': np.arange(self.length),
            'data': series,
            'stationary': (np.ones(self.length)).astype(int),
            'seasonal': np.zeros(self.length).astype(int),

        })
        return df, info

    #ANOMALIES    

    def generate_point_anomaly(self, df, location=None, scale_factor=1, is_spike=True, is_loc = None):
        series = df['data'].copy()
        n = len(series)
        num_anomalies = 1
    
        # Determine candidate indices based on location
        if location == "beginning":
            candidate_range = np.arange(int(0.1 * n), int(0.3 * n))
        elif location == "middle":
            candidate_range = np.arange(int(0.4 * n), int(0.6 * n))
        elif location == "end":
            candidate_range = np.arange(int(0.7 * n), int(0.9 * n))
        else:
            candidate_range = np.arange(int(0.1 * n), int(0.9 * n))  # Default safe zone
    
        if len(candidate_range) == 0:
            raise ValueError("No valid candidate indices found for the given location.")
    
        # Select point anomaly index
        anomaly_indices = np.random.choice(candidate_range, num_anomalies, replace=False)
    
        # Inject anomaly guaranteed to be dominant
        for idx in anomaly_indices:
            local_std = np.std(series[max(0, idx - int(n*0.5)):min(n, idx + int(n*0.5))])
            global_spike = np.max(np.abs(series - np.mean(series)))
            global_spike_factor = np.random.uniform(1.1,1.3)
            if is_spike:
                magnitude = global_spike_factor * global_spike * scale_factor
            else:
                magnitude = local_std * np.random.uniform(1.5, 2.5) * scale_factor
            direction = np.random.choice([-1, 1])
            series[idx] = np.mean(series) + direction * magnitude
    
        info = {'type': 'anomaly', 'subtype': 'single_point', 'num_anomalies': num_anomalies, 'anomaly_indices': anomaly_indices, 'location': location}
    
        df.loc[:, 'data'] = series
        df.loc[:, 'stationary'] = 0
        df.loc[:, 'point_anom_single'] = 1

        if is_loc:
            point_anom_label = np.zeros(n, dtype=int)
            point_anom_label[anomaly_indices] = 1
            df.loc[:, "point_anom_label"] = point_anom_label

        return df, info

    def generate_point_anomalies(self, df, scale_factor=1,is_loc=None):
        series = df['data'].copy()
        n = len(series)

        def compute_point_anomaly_count(length):
            min_anom = 2
            max_anom = min(40, int(length * 0.02))

            if max_anom <= min_anom:
                return min_anom
            return np.random.randint(min_anom, max_anom + 1)
    
        # Determine how many anomalies to inject
        num_anomalies = compute_point_anomaly_count(n)
    
        # Select point anomaly indices
        anomaly_indices = np.random.choice(n, num_anomalies, replace=False)
        anomaly_indices = np.sort(anomaly_indices)
    
        # Compute the max deviation from the mean — natural peak size
        global_spike = np.max(np.abs(series - np.mean(series)))
        for idx in anomaly_indices:
            local_window = series[max(0, idx - int(n*0.5)):min(n, idx + int(n*0.5))]
            local_std = np.std(local_window)
    
            # Choose base magnitude using local std with randomness
            base_mag = local_std * np.random.uniform(2, 3.5)
    
            # Enforce visibility: must be at least 1.1× natural spike
            global_spike_factor = np.random.uniform(0.5,1.2)
            min_visible_mag = global_spike_factor * global_spike
            magnitude = max(base_mag, min_visible_mag) * scale_factor
            
            # Add anomaly
            direction = np.random.choice([-1, 1])
            series[idx] = np.mean(series) + direction * magnitude
    
        info = {'type': 'anomaly', 'subtype': 'multiple_point','num_anomalies': num_anomalies, 'anomaly_indices': anomaly_indices}
    
        df.loc[:, 'data'] = series
        df.loc[:, 'stationary'] = 0
        df.loc[:, 'point_anom_multi'] = 1

        if is_loc:
            point_anom_label = np.zeros(n, dtype=int)
            point_anom_label[anomaly_indices] = 1
            df.loc[:, "point_anom_label"] = point_anom_label

        return df, info

    def generate_collective_anomalies(
        self,
        df,
        num_anomalies=1,
        location="middle",
        scale_factor=1,
        anomaly_shapes="rectangular",
        edge_margin=0.05,
        min_distance=0.10,
        max_attempts=1000,
        is_loc = None,
    ):
        series = df["data"].copy()
        original_series = series.copy()
        n = len(series)

        shape_configs = {
            "rectangular": {
                "length_range": (0.05, 0.09),
                "magnitude_range": (1, 1.75),
                "residual_weight": None,
                "method": "add"
            },
            "gaussian": {
                "length_range": (0.09, 0.15),
                "magnitude_range": (1.5, 2.5),
                "residual_weight": 0.1,
                "method": "baseline"
            },
            "triangular": {
                "length_range": (0.09, 0.15),
                "magnitude_range": (1.5, 2.5),
                "residual_weight": 0.15,
                "method": "baseline"
            },
            "ramp": {
                "length_range": (0.05, 0.1),
                "magnitude_range": (1.5, 2.5),
                "residual_weight": 0.15,
                "method": "baseline"
            },
            "decay": {
                "length_range": (0.05, 0.1),
                "magnitude_range": (1.5, 2.5),
                "residual_weight": 0.15,
                "method": "baseline"
            }
            }
        

        valid_shapes = list(shape_configs.keys())

        # If a string is given, use the same shape for all anomalies.
        if isinstance(anomaly_shapes, str):
            if anomaly_shapes not in valid_shapes:
                raise ValueError(f"Unknown anomaly shape: {anomaly_shapes}. Valid shapes are: {valid_shapes}")

            anomaly_shapes = [anomaly_shapes] * num_anomalies

        elif isinstance(anomaly_shapes, list):
            if len(anomaly_shapes) == 0:
                raise ValueError("anomaly_shapes list cannot be empty.")

            for shape in anomaly_shapes:
                if shape not in valid_shapes:
                    raise ValueError(f"Unknown anomaly shape: {shape}. Valid shapes are: {valid_shapes}")

            # Case 1: one shape in a list -> repeat it for all anomalies
            if len(anomaly_shapes) == 1:
                anomaly_shapes = anomaly_shapes * num_anomalies

            # Case 2: one shape per anomaly -> use directly
            elif len(anomaly_shapes) == num_anomalies:
                anomaly_shapes = anomaly_shapes

            # Case 3: mismatch -> raise error
            else:
                raise ValueError(
                    f"When anomaly_shapes is a list, it must either contain exactly 1 shape "
                    f"or match num_anomalies. Got {len(anomaly_shapes)} shapes for "
                    f"{num_anomalies} anomalies."
                )

        else:
            raise TypeError("anomaly_shapes must be either a string or a list of strings.")

        edge_margin_points = int(edge_margin * n)
        min_distance_points = int(min_distance * n)

        if num_anomalies > 1:
            location_used = "none"
        else:
            location_used = location

        def get_shape_profile(length, shape):
            if length <= 1:
                return np.ones(length)

            if shape == "rectangular":
                return np.ones(length)

            x = np.linspace(0, 1, length)

            if shape == "gaussian":
                center = 0.5
                width = 0.28
                profile = np.exp(-0.5 * ((x - center) / width) ** 2)
                profile = profile - profile.min()
                profile = profile / np.max(profile)

            elif shape == "triangular":
                profile = 1 - np.abs(2 * x - 1)
                profile = profile ** 1.5

            elif shape == "ramp":
                profile = x

            elif shape == "decay":
                profile = np.linspace(1, 0, length)

            else:
                raise ValueError(f"Unknown anomaly shape: {shape}")

            return profile

        def get_start_bounds(location_used, length):
            if location_used == "beginning":
                start_low = int(0.10 * n)
                start_high = int(0.30 * n)

            elif location_used == "middle":
                start_low = int(0.40 * n)
                start_high = int(0.60 * n)

            elif location_used == "end":
                start_low = int(0.70 * n)
                start_high = int(0.90 * n)

            else:
                start_low = int(0.10 * n)
                start_high = int(0.85 * n)

            latest_possible_start = n - edge_margin_points - length

            start_low = max(start_low, edge_margin_points)
            start_high = min(start_high, latest_possible_start)

            return start_low, start_high

        def interval_is_valid(start, end, selected_intervals):
            for existing_start, existing_end in selected_intervals:
                too_close_or_overlapping = not (
                    end + min_distance_points <= existing_start
                    or start >= existing_end + min_distance_points
                )

                if too_close_or_overlapping:
                    return False

            return True

        selected_intervals = []
        records = []

        for shape in anomaly_shapes:
            config = shape_configs[shape]

            min_len = max(3, int(config["length_range"][0] * n))
            max_len = max(min_len + 1, int(config["length_range"][1] * n))

            found_interval = False

            for _ in range(max_attempts):
                length = np.random.randint(min_len, max_len + 1)

                start_low, start_high = get_start_bounds(location_used, length)

                if start_high <= start_low:
                    continue

                start = np.random.randint(start_low, start_high + 1)
                end = start + length

                # First check overlap / distance condition
                if not interval_is_valid(start, end, selected_intervals):
                    continue

                # Reject visually awkward boundaries.
                # This prevents the anomaly from starting or ending exactly at an extreme jump/spike.
                boundary_window = max(5, int(0.05 * n))
                boundary_threshold = 2.5

                left = max(0, start - boundary_window)
                right = min(n, end + boundary_window)

                local_region = original_series.iloc[left:right].to_numpy()
                local_std = np.std(local_region)

                if local_std < 1e-8:
                    local_std = np.std(original_series.to_numpy())

                if local_std < 1e-8:
                    local_std = 1.0

                start_jump = abs(original_series.iloc[start] - original_series.iloc[start - 1]) if start > 0 else 0
                end_jump = abs(original_series.iloc[end] - original_series.iloc[end - 1]) if end < n else 0

                if start_jump > boundary_threshold * local_std:
                    continue

                if end_jump > boundary_threshold * local_std:
                    continue

                selected_intervals.append((start, end))
                found_interval = True
                break

            if not found_interval:
                raise ValueError(
                    f"Could not place anomaly with shape '{shape}'. "
                    f"Try reducing num_anomalies, min_distance, or anomaly length ranges."
                )

            profile = get_shape_profile(length, shape)

            local_start = max(0, start - int(0.10 * n))
            local_segment = original_series.iloc[local_start:start].to_numpy()

            if len(local_segment) > 3 and np.std(local_segment) > 1e-8:
                local_std = np.std(local_segment)
            else:
                local_std = np.std(original_series.to_numpy())

            if local_std < 1e-8:
                local_std = 1.0

            magnitude_strength = np.random.uniform(*config["magnitude_range"])
            magnitude = magnitude_strength * local_std * scale_factor
            sign = np.random.choice([-1, 1])

            anomaly_pattern = sign * magnitude * profile

            if config["method"] == "add":
                segment = series.iloc[start:end].to_numpy()
                series.iloc[start:end] = segment + anomaly_pattern

            elif config["method"] == "baseline":
                segment = series.iloc[start:end].to_numpy()

                baseline_window = max(5, int(0.03 * n))

                before_segment = series.iloc[
                    max(0, start - baseline_window):start
                ]

                after_segment = series.iloc[
                    end:min(n, end + baseline_window)
                ]

                if len(before_segment) > 0:
                    baseline_start = np.median(before_segment)
                else:
                    baseline_start = series.iloc[start]

                if len(after_segment) > 0:
                    baseline_end = np.median(after_segment)
                else:
                    baseline_end = series.iloc[end - 1]

                baseline = np.linspace(
                    baseline_start,
                    baseline_end,
                    length
                )

                segment_trend = np.linspace(
                    segment[0],
                    segment[-1],
                    length
                )

                residual = segment - segment_trend
                residual_weight = config["residual_weight"]

                series.iloc[start:end] = (
                    baseline
                    + residual_weight * residual
                    + anomaly_pattern
                )

            # Record the anomaly after it has been created
            records.append({
                "start": start,
                "end": end,
                "shape": shape,
                "magnitude": sign * magnitude,
                "magnitude_strength": magnitude_strength,
                "length": length
            })

        # This part must be outside the anomaly_shapes loop
        records = sorted(
            records,
            key=lambda item: item["start"]
        )

        selected_starts = np.array(
            [item["start"] for item in records],
            dtype=int
        )

        ends = np.array(
            [item["end"] for item in records],
            dtype=int
        )

        shapes_used = [
            item["shape"]
            for item in records
        ]

        magnitudes = [
            item["magnitude"]
            for item in records
        ]

        lengths = [
            item["length"]
            for item in records
        ]

        magnitude_strengths = [
            item["magnitude_strength"]
            for item in records
        ]

        info = {
            "type": "anomaly",
            "subtype": "collective",
            "anomaly_shapes": shapes_used,
            "num_anomalies": len(records),
            "location": location_used,
            "starts": selected_starts,
            "ends": ends,
            "lengths": lengths,
            "magnitudes": magnitudes,
            "magnitude_strengths": magnitude_strengths
        }

        df.loc[:, "data"] = series
        df.loc[:, "stationary"] = 0
        df.loc[:, "collect_anom"] = 1

        if is_loc is True:
            collect_anom_label = np.zeros(
                n,
                dtype=int
            )

            for record in records:
                start = int(record["start"])
                end = int(record["end"])

                collect_anom_label[start:end] = 1

            df.loc[:, "collect_anom_label"] = (
                collect_anom_label
            )

        return df, info
    
    def generate_contextual_anomalies(
        self,
        df,
        num_anomalies=1,
        location=None,
        scale_factor=1,
        anomaly_strength=1,
        seasonal_period=None,
        max_attempts=10,
        is_loc=None
    ):
        series_original = df["data"].copy()
        n = len(series_original)

        for attempt in range(max_attempts):
            min_distance = max(
                1,
                int((0.05 - attempt * 0.003) * n)
            )

            series = series_original.copy()

            # These are the selected peak/valley center points.
            selected_starts = []

            # These store the actual anomaly intervals.
            anomaly_intervals = []

            # Decide the seasonal period
            if seasonal_period is not None:
                period = seasonal_period
                generate_seasonality = False
            else:
                min_period = max(5, n // 20)
                max_period = n // 6

                allowed = [5, 7, 12, 24, 30, 52, 90, 180]

                periods = [
                    p for p in allowed
                    if min_period <= p <= max_period
                ]

                if not periods:
                    continue

                period = random.choice(periods)
                generate_seasonality = True

            # Generate or estimate seasonality
            if generate_seasonality:
                amplitude = (
                    np.std(series)
                    * np.random.uniform(1.5, 3)
                )

                seasonality = (
                    amplitude
                    * np.sin(
                        2 * np.pi * np.arange(n) / period
                    )
                )

                series += seasonality * scale_factor

            else:
                seasonality = np.sin(
                    2 * np.pi * np.arange(n) / period
                )

            # Find contextual points from a clean sine wave
            pure_seasonality = np.sin(
                2 * np.pi * np.arange(n) / period
            )

            peaks = np.where(
                (
                    pure_seasonality[1:-1]
                    > pure_seasonality[:-2]
                )
                &
                (
                    pure_seasonality[1:-1]
                    > pure_seasonality[2:]
                )
            )[0] + 1

            valleys = np.where(
                (
                    pure_seasonality[1:-1]
                    < pure_seasonality[:-2]
                )
                &
                (
                    pure_seasonality[1:-1]
                    < pure_seasonality[2:]
                )
            )[0] + 1

            candidate_indices = np.concatenate(
                [peaks, valleys]
            )

            # Determine candidate regions
            if num_anomalies == 1:
                if location == "beginning":
                    candidate_range = np.arange(
                        int(0.1 * n),
                        int(0.3 * n)
                    )

                elif location == "middle":
                    candidate_range = np.arange(
                        int(0.4 * n),
                        int(0.6 * n)
                    )

                elif location == "end":
                    candidate_range = np.arange(
                        int(0.7 * n),
                        int(0.9 * n)
                    )

                else:
                    candidate_range = np.arange(
                        int(0.1 * n),
                        int(0.85 * n)
                    )

                    location = "none"

            else:
                candidate_range = np.arange(
                    int(0.1 * n),
                    int(0.85 * n)
                )

                location = "none"

            candidate_indices = np.array([
                i
                for i in candidate_indices
                if i in candidate_range
            ])

            if len(candidate_indices) == 0:
                print(
                    f"[Attempt {attempt + 1}] "
                    f"No candidates found for "
                    f"n={n}, period={period}"
                )
                continue

            # Select anomaly centers with spacing
            candidates = candidate_indices.copy()
            np.random.shuffle(candidates)

            for center in candidates:
                if all(
                    abs(center - previous_center)
                    >= min_distance
                    for previous_center in selected_starts
                ):
                    selected_starts.append(center)

                if len(selected_starts) == num_anomalies:
                    break

            # Fill remaining anomalies without spacing if necessary
            if len(selected_starts) < num_anomalies:
                remaining = list(
                    set(candidate_indices)
                    - set(selected_starts)
                )

                np.random.shuffle(remaining)

                for center in remaining:
                    selected_starts.append(center)

                    if len(selected_starts) == num_anomalies:
                        break

            if len(selected_starts) == 0:
                continue

            # Apply contextual anomalies
            for center in selected_starts:
                anomaly_length = min(
                    max(int(period * 0.5), 10),
                    int(0.2 * n)
                )

                start = max(
                    0,
                    center - anomaly_length // 2
                )

                end = min(
                    n,
                    start + anomaly_length
                )

                # Store the actual anomaly interval.
                anomaly_intervals.append((start, end))

                local_season = seasonality[start:end]

                series.iloc[start:end] -= (
                    2
                    * local_season
                    * anomaly_strength
                )

            # Successful generation
            break

        else:
            print(
                f"generate_contextual_anomalies "
                f"failed for n={n}"
            )

            return df, None

        # Sort anomaly intervals according to their start positions
        anomaly_intervals = sorted(
            anomaly_intervals,
            key=lambda interval: interval[0]
        )

        anomaly_starts = np.array([
            start
            for start, end in anomaly_intervals
        ])

        anomaly_ends = np.array([
            end
            for start, end in anomaly_intervals
        ])

        info = {
            "type": "anomaly",
            "subtype": "contextual",
            "num_anomalies": len(anomaly_intervals),
            "location": location,
            "starts": anomaly_starts,
            "ends": anomaly_ends
        }

        df.loc[:, "data"] = series
        df.loc[:, "stationary"] = 0
        df.loc[:, "context_anom"] = 1
        df.loc[:, "seasonal"] = 1

        # Create location labels only when requested
        if is_loc is True:
            context_anom_label = np.zeros(
                n,
                dtype=int
            )

            for start, end in anomaly_intervals:
                context_anom_label[start:end] = 1

            df.loc[
                :,
                "context_anom_label"
            ] = context_anom_label

        return df, info

    #TRENDS - DETERMINISTIC TRENDS

    def generate_deterministic_trend_linear(self, df, sign = None, slope= None, noise_std = None, intercept = 1, scale_factor = 1):
        series = df['data'].copy()
        sign = sign if sign is not None else np.random.choice([-1,1])
        noise_std = noise_std if noise_std is not None else np.random.uniform(0.1, 1.5)
        slope = slope if slope is not None else sign * random.uniform(0.05, 0.5) / (len(series) / 100)
        trend = intercept + slope * np.arange(len(series)) + np.random.normal(0, noise_std, len(series))
        series += trend * scale_factor
        info = {'type' : 'trend', 'subtype': 'deterministic_linear', 'sign': sign, 'slope': slope, 'intercept': intercept}
        df.loc[:,'data'] = series
        df.loc[:,'stationary'] = 0
        if sign > 0:
            df.loc[:,'det_lin_up'] = 1
        else:
            df.loc[:,'det_lin_down'] = 1
        return df, info

    def generate_deterministic_trend_quadratic(self, df, sign=None, a=None, b=None, c=None,noise_std=None, scale_factor=1,asymmetric=False, location="center"):
        series = df['data'].copy()
        sign = sign if sign in [-1, 1] else random.choice([-1, 1])
        length = len(series)
        t = np.linspace(-1, 1, length)
    
        # Choose strength of curvature
        a = a if a is not None else sign * random.uniform(2.0, 5.0)
    
        # Compute linear term to move vertex
        if location == "center":
            b = 0
        elif location == "left":
            b = -2 * a * (-0.5)  # vertex at t = -0.5
        elif location == "right":
            b = -2 * a * (0.5)   # vertex at t = +0.5
        else:
            raise ValueError("location must be 'center', 'left', or 'right'")
    
        c = c if c is not None else 0
    
        trend = (a * t**2 + b * t + c) * scale_factor
    
        noise_std = noise_std if noise_std is not None else np.random.uniform(0.001, 0.01)
        noise = np.random.normal(0, noise_std, length)

        info = {'type' : 'trend', 'subtype': 'deterministic_quadratic','sign': sign, 'a': a, 'b': b, 'c': c}
    
        series += trend + noise
        df.loc[:, 'data'] = series
        df.loc[:, 'stationary'] = 0
        df.loc[:, 'det_quad'] = 1
        return df, info

    def generate_deterministic_trend_cubic(self, df, sign=None, amplitude=10, noise_std=None,scale_factor=1, asymmetric=False, location="center"):
        series = df['data'].copy()
        sign = sign if sign in [-1, 1] else random.choice([-1, 1])
        length = len(series)
        t = np.linspace(-1, 1, length)
    
        a = 1.0  # fixed cubic term
        c = -1.0  # linear slope for S shape
    
        # Inflection point: t_i = -b / (3a) → solve for b
        if location == "center":
            b = 0
        elif location == "left":
            b = -3 * a * (-0.5)  # inflection at t = -0.5
        elif location == "right":
            b = -3 * a * (0.5)   # inflection at t = +0.5
        else:
            raise ValueError("location must be 'center', 'left', or 'right'")
    
        # If asymmetric override is also set, add to b
        if asymmetric:
            b += sign * random.uniform(0.5, 2.0)
    
        # Final trend
        trend = (a * t**3 + b * t**2 + c * t) * amplitude
    
        noise_std = noise_std if noise_std is not None else np.random.uniform(0.01, 0.05)
        noise = np.random.normal(0, noise_std, length)
    
        series += trend * scale_factor + noise

        info = {'type' : 'trend', 'subtype': 'deterministic_cubic','sign': sign, 'a': a, 'b': b}
        
        df.loc[:, 'data'] = series
        df.loc[:, 'stationary'] = 0
        df.loc[:, 'det_cubic'] = 1
        return df, info

    def generate_deterministic_trend_exponential(self, df, sign=None, a=None, b=None, noise_std=None, scale_factor=1):
        series = df['data'].copy()
        sign = sign if sign in [-1, 1] else random.choice([-1, 1])
        length = len(series)
        a = a if a is not None else random.uniform(1.0, 2.0)
        b = b if b is not None else random.uniform(1.5, 3.0)
        t = np.linspace(0, 2, len(series))

        if sign == 1:
            noise_std = noise_std if noise_std is not None else np.random.uniform(0.1, 0.5)
            trend = a * np.exp(b * t)
            scale_factor = 1
        else:
            noise_std = noise_std if noise_std is not None else np.random.uniform(0.01, 0.05)
            trend = a * np.exp(-b * t)
            scale_factor = 5
            
        trend *= scale_factor
        noise = np.random.normal(0, noise_std, length)
    
        series += trend + noise*3

        info = {'type' : 'trend', 'subtype': 'deterministic_exponential','sign': sign, 'a': a, 'b': b}
    
        df.loc[:, 'data'] = series
        df.loc[:, 'stationary'] = 0
        df.loc[:, 'det_exp'] = 1
        return df, info

    def generate_deterministic_trend_damped(self, df, sign=None, a=None, b=None, damping_rate=None, noise_std=None, scale_factor=1):
        series = df['data'].copy()
        noise_std = noise_std if noise_std is not None else np.random.uniform(0.1, 1.5)
        sign = sign if sign is not None else random.choice([-1, 1])
        a = a if a is not None else sign * np.random.normal(loc=1.0, scale=0.2)
        b = b if b is not None else np.random.normal(loc=0.1, scale=0.05)
        damping_rate = damping_rate if damping_rate is not None else random.uniform(0.01, 0.005)
        t = np.arange(len(series))
        noise = np.random.normal(0, noise_std, len(series))
        trend = (a * t + b) * np.exp(-damping_rate * t) * scale_factor + noise
        series += trend
        info = {'type' : 'trend', 'subtype': 'deterministic_damped','damping_rate': damping_rate, 'a': a, 'b': b}
        df.loc[:, 'data'] = series
        df.loc[:,'stationary'] = 0
        df.loc[:, 'det_damped'] = 1
        return df,info

    #TRENDS - STOCHASTIC TRENDS

    def generate_stochastic_trend(self, kind='rw', d = 1, const=False, drift=None, noise_std=1.0):
        t = np.arange(self.length)
        noise = np.random.normal(0, noise_std, self.length)
    
        if kind == 'rw':
            info = {'type': 'trend', 'subtytpe': 'random_walk'}
            series = np.cumsum(noise)

        elif kind == 'rwd':
            if drift is None:
                drift = np.random.uniform(0.01, 0.1)
            info = {'type': 'trend', 'subtype': 'random_walk_with_drift', 'drift': drift}
            series = drift * t + np.cumsum(noise)
    
        elif kind == 'ari':
            series, info = self.generate_ari_series(length=self.length, d = d, const = const)
    
        elif kind == 'ima':
            series, info = self.generate_ima_series(length=self.length, d = d, const = const)
    
        elif kind == 'arima':
            series, info = self.generate_arima_series(length=self.length, d = d, const = const)
    
        else:
            raise ValueError("Invalid kind. Choose from 'rw', 'rwd', 'ari', 'ima', or 'arima'.")

        df = pd.DataFrame({
            'time': np.arange(self.length),
            'data': series,
            'stationary': (np.zeros(self.length)).astype(int),
            'seasonal': np.zeros(self.length).astype(int),
        })
        return df, info

    # PERIOD HELPERS

    def get_calendar_periods(self):
        """
        Calendar-meaningful seasonal periods for different sampling frequencies.
        Period always means: number of observations per cycle.
        """
        return {
            "monthly": {
                3: "quarterly cycle",
                6: "semiannual cycle",
                12: "annual cycle"
            },
            "quarterly": {
                4: "annual cycle"
            },
            "daily": {
                7: "weekly cycle",
                30: "monthly-ish cycle",
                90: "quarterly-ish cycle",
                180: "semiannual-ish cycle",
                365: "annual cycle"
            },
            "weekly": {
                4: "monthly-ish cycle",
                13: "quarterly cycle",
                26: "semiannual cycle",
                52: "annual cycle"
            },
            "business_daily": {
                5: "weekly cycle",
                21: "monthly-ish cycle",
                63: "quarterly-ish cycle",
                126: "semiannual-ish cycle",
                252: "annual-ish cycle"
            },
            "hourly": {
                24: "daily cycle",
                168: "weekly cycle"
            }
        }

    def get_all_calendar_periods(self):
        calendar_periods = self.get_calendar_periods()

        all_periods = sorted(
            set(
                period
                for sampling_dict in calendar_periods.values()
                for period in sampling_dict.keys()
            )
        )

        return all_periods

    def get_period_meanings(self, period):
        """
        Returns all possible calendar interpretations of a period.
        Example:
            period=4 can mean:
            - quarterly data: annual cycle
            - weekly data: monthly-ish cycle
        """
        calendar_periods = self.get_calendar_periods()
        meanings = []

        for sampling_frequency, period_dict in calendar_periods.items():
            if period in period_dict:
                meanings.append({
                    "sampling_frequency": sampling_frequency,
                    "meaning": period_dict[period]
                })

        return meanings

    def get_valid_calendar_periods(
        self,
        allowed_periods=None,
        min_cycles=6
    ):
        """
        Filters periods according to series length.

        min_cycles=6 means:
            selected period should appear at least about 6 times in the series.
        """
        n = self.length

        if allowed_periods is None:
            allowed_periods = self.get_all_calendar_periods()

        allowed_periods = sorted(set(int(p) for p in allowed_periods))

        max_period = n // min_cycles

        valid_periods = [
            p for p in allowed_periods
            if p <= max_period
        ]

        return valid_periods

    def choose_calendar_period(
        self,
        period=None,
        allowed_periods=None,
        min_cycles=6
    ):
        """
        Chooses or validates a period using calendar-meaningful periods.
        """
        n = self.length

        if allowed_periods is None:
            allowed_periods = self.get_all_calendar_periods()

        allowed_periods = sorted(set(int(p) for p in allowed_periods))
        valid_periods = self.get_valid_calendar_periods(
            allowed_periods=allowed_periods,
            min_cycles=min_cycles
        )

        if len(valid_periods) == 0:
            raise ValueError(
                f"No valid period found for length={n}. "
                f"Allowed periods are {allowed_periods}, but min_cycles={min_cycles} requires period <= {n // min_cycles}."
            )

        if period is None:
            period = random.choice(valid_periods)
        else:
            period = int(period)

            if period not in allowed_periods:
                raise ValueError(
                    f"period={period} is not in allowed calendar periods: {allowed_periods}"
                )

            if period not in valid_periods:
                raise ValueError(
                    f"period={period} is too large for length={n} with min_cycles={min_cycles}. "
                    f"Valid periods are {valid_periods}."
                )

        return period, valid_periods

    def normalize_period_list(self, periods):
        """
        Converts period input into a list.
        """
        if periods is None:
            return None

        if isinstance(periods, (int, np.integer)):
            return [int(periods)]

        return [int(p) for p in periods]

# SEASONALITY

    def generate_single_seasonality(
        self,
        period=None,
        amplitude=None,
        noise_std=None,
        scale_factor=1,
        num_harmonics=1,
        allowed_periods=None,
        min_cycles=6
    ):
        n = self.length
        t = np.arange(n)

        series = np.random.normal(loc=0.0, scale=0.2, size=n)

        noise_std = noise_std if noise_std is not None else np.random.uniform(0.01, 0.05)

        period, valid_periods = self.choose_calendar_period(
            period=period,
            allowed_periods=allowed_periods,
            min_cycles=min_cycles
        )

        base_std = np.std(series)
        amplitude = amplitude if amplitude is not None else base_std * np.random.uniform(0.5, 2.5)

        seasonality = np.zeros(n)
        coefficients = []

        for k in range(1, num_harmonics + 1):
            A_k = amplitude * np.random.uniform(0.5, 1.0) / k
            B_k = amplitude * np.random.uniform(0.5, 1.0) / k

            seasonality += A_k * np.sin(2 * np.pi * k * t / period)
            seasonality += B_k * np.cos(2 * np.pi * k * t / period)

            coefficients.append({
                "harmonic": k,
                "sin_coef": A_k,
                "cos_coef": B_k
            })

        seasonality += np.random.normal(0, noise_std, size=n)

        series += seasonality * scale_factor

        df = pd.DataFrame({
            "time": np.arange(n),
            "data": series,
            "stationary": np.zeros(n).astype(int),
            "seasonal": np.ones(n).astype(int),
            "single_seas": np.ones(n).astype(int)
        })

        info = {
            "type": "seasonal",
            "subtype": "single_seasonality",
            "periods": [period],
            "period_meanings": {
                period: self.get_period_meanings(period)
            },
            "amplitude": amplitude,
            "noise_std": noise_std,
            "scale_factor": scale_factor,
            "num_harmonics": num_harmonics,
            "coefficients": coefficients
        }

        return df, info
    
    def generate_multiple_seasonality(
        self,
        num_components=2,
        periods=None,
        amplitudes=None,
        noise_std=None,
        scale_factor=3,
        num_harmonics=1,
        allowed_periods=None,
        min_cycles=6
    ):
        n = self.length
        t = np.arange(n)

        series = np.random.normal(loc=0.0, scale=0.2, size=n)

        noise_std = noise_std if noise_std is not None else np.random.uniform(0.01, 0.05)

        if allowed_periods is None:
            allowed_periods = self.get_all_calendar_periods()

        allowed_periods = sorted(set(int(p) for p in allowed_periods))

        valid_periods = self.get_valid_calendar_periods(
            allowed_periods=allowed_periods,
            min_cycles=min_cycles
        )

        if len(valid_periods) < 2 and periods is None:
            raise ValueError(
                f"Multiple seasonality needs at least 2 valid periods. "
                f"For length={n}, valid periods are {valid_periods}."
            )

        periods = self.normalize_period_list(periods)

        if periods is None:
            periods = random.sample(
                valid_periods,
                min(num_components, len(valid_periods))
            )
        else:
            for p in periods:
                if p not in allowed_periods:
                    raise ValueError(
                        f"period={p} is not in allowed calendar periods: {allowed_periods}"
                    )

                if p not in valid_periods:
                    raise ValueError(
                        f"period={p} is too large for length={n} with min_cycles={min_cycles}. "
                        f"Valid periods are {valid_periods}."
                    )

            if len(periods) < 2:
                raise ValueError(
                    "Multiple seasonality should have at least 2 periods. "
                    "Pass something like periods=[7, 30] or periods=[12, 24]."
                )

        if amplitudes is None:
            base_std = np.std(series)
            amplitudes = [
                base_std * np.random.uniform(0.5, 2.0)
                for _ in periods
            ]
        else:
            if len(amplitudes) != len(periods):
                raise ValueError(
                    f"Length of amplitudes must match length of periods. "
                    f"Got {len(amplitudes)} amplitudes and {len(periods)} periods."
                )

        seasonality = np.zeros(n)

        periods_meta = []
        amplitudes_meta = []
        coefficients_meta = []

        for period, amplitude in zip(periods, amplitudes):
            period_coefficients = []

            for k in range(1, num_harmonics + 1):
                A_k = amplitude * np.random.uniform(0.5, 1.0) / k
                B_k = amplitude * np.random.uniform(0.5, 1.0) / k

                seasonality += A_k * np.sin(2 * np.pi * k * t / period)
                seasonality += B_k * np.cos(2 * np.pi * k * t / period)

                period_coefficients.append({
                    "harmonic": k,
                    "sin_coef": A_k,
                    "cos_coef": B_k
                })

            periods_meta.append(period)
            amplitudes_meta.append(amplitude)
            coefficients_meta.append({
                "period": period,
                "coefficients": period_coefficients
            })

        seasonality += np.random.normal(0, noise_std, size=n)

        series += seasonality * scale_factor

        df = pd.DataFrame({
            "time": np.arange(n),
            "data": series,
            "stationary": np.zeros(n).astype(int),
            "seasonal": np.ones(n).astype(int),
            "multiple_seas": np.ones(n).astype(int)
        })

        info = {
            "type": "seasonal",
            "subtype": "multiple_seasonality",
            "periods": periods_meta,
            "period_meanings": {
                p: self.get_period_meanings(p)
                for p in periods_meta
            },
            "amplitudes": amplitudes_meta,
            "noise_std": noise_std,
            "scale_factor": scale_factor,
            "num_harmonics": num_harmonics,
            "coefficients": coefficients_meta
        }

        return df, info

    def generate_sarima_series(
        self,
        period=None,
        amplitude=None,
        noise_std=None,
        scale_factor=1,
        initial_std=0.2,
        num_harmonics=1,
        allowed_periods=None,
        min_cycles=6
    ):
        """
        Seasonal unit root case with deterministic Fourier seasonal difference.

        Model:
            Y_t - Y_{t-s} = Fourier(t) + noise
        """

        n = self.length
        t = np.arange(n)

        period, _ = self.choose_calendar_period(
            period=period,
            allowed_periods=allowed_periods,
            min_cycles=min_cycles
        )

        if n <= period:
            raise ValueError("Series length must be larger than the seasonal period.")

        noise_std = noise_std if noise_std is not None else np.random.uniform(0.1, 0.20)

        if amplitude is None:
            base_series = np.random.normal(loc=0.0, scale=0.2, size=n)
            amplitude = np.std(base_series) * np.random.uniform(0.5, 2.5)

        seasonal_difference = np.zeros(n)
        coefficients = []

        for k in range(1, num_harmonics + 1):
            A_k = amplitude * np.random.uniform(0.5, 1.0) / k
            B_k = amplitude * np.random.uniform(0.5, 1.0) / k

            seasonal_difference += A_k * np.sin(2 * np.pi * k * t / period)
            seasonal_difference += B_k * np.cos(2 * np.pi * k * t / period)

            coefficients.append({
                "harmonic": k,
                "sin_coef": A_k,
                "cos_coef": B_k
            })

        seasonal_difference += np.random.normal(0, noise_std, size=n)
        seasonal_difference = seasonal_difference * scale_factor

        series = np.zeros(n)
        series[:period] = np.random.normal(0, initial_std, size=period)

        for i in range(period, n):
            series[i] = series[i - period] + seasonal_difference[i]

        actual_seasonal_difference = np.full(n, np.nan)
        actual_seasonal_difference[period:] = series[period:] - series[:-period]

        df = pd.DataFrame({
            "time": np.arange(n),
            "data": series,
            "seasonal_diff": actual_seasonal_difference,
            "stationary": np.zeros(n).astype(int),
            "seasonal": np.ones(n).astype(int),
            "sarima": np.ones(n).astype(int)
        })

        info = {
            "type": "seasonal",
            "subtype": "SARIMA",
            "periods": [period],
            "period_meanings": {
                period: self.get_period_meanings(period)
            },
            "diff": 0,
            "seasonal_diff": 1,
            "unit_root": "seasonal_unit_root",
            "amplitude": amplitude,
            "noise_std": noise_std,
            "scale_factor": scale_factor,
            "initial_std": initial_std,
            "num_harmonics": num_harmonics,
            "coefficients": coefficients
        }

        return df, info

    def generate_deterministic_sarma_series(
        self,
        period=None,
        amplitude=None,
        noise_std=None,
        scale_factor=1,
        error_scale=None,
        num_harmonics=1,
        order_range=(1, 2),
        seasonal_order_range=(1, 1),
        coef_range=(-0.4, 0.4),
        seasonal_coef_range=(-0.4, 0.4),
        allowed_periods=None,
        min_cycles=6,
        max_attempts=1000
    ):
        """
        Generates a SARMA-like seasonal series with deterministic seasonality.

        Model:
            Y_t = f_t + u_t + e_t

        where:
            f_t = deterministic Fourier seasonality
            u_t = stationary SARMA error process
            e_t = small observation noise

        There is no unit root and no differencing.
        """

        n = self.length
        t = np.arange(n)

        period, _ = self.choose_calendar_period(
            period=period,
            allowed_periods=allowed_periods,
            min_cycles=min_cycles
        )

        s = period

        if amplitude is None:
            base_series = np.random.normal(loc=0.0, scale=0.2, size=n)
            amplitude = np.std(base_series) * np.random.uniform(0.8, 2.5)

        deterministic_seasonality = np.zeros(n)
        fourier_coefficients = []

        for k in range(1, num_harmonics + 1):
            A_k = amplitude * np.random.uniform(0.5, 1.0) / k
            B_k = amplitude * np.random.uniform(0.5, 1.0) / k

            deterministic_seasonality += A_k * np.sin(2 * np.pi * k * t / s)
            deterministic_seasonality += B_k * np.cos(2 * np.pi * k * t / s)

            fourier_coefficients.append({
                "harmonic": k,
                "sin_coef": A_k,
                "cos_coef": B_k
            })

        deterministic_seasonality = deterministic_seasonality * scale_factor

        sarma_error = None

        for _ in range(max_attempts):
            ar_order = np.random.randint(order_range[0], order_range[1] + 1)
            ma_order = np.random.randint(order_range[0], order_range[1] + 1)

            seasonal_ar_order = np.random.randint(
                seasonal_order_range[0],
                seasonal_order_range[1] + 1
            )
            seasonal_ma_order = np.random.randint(
                seasonal_order_range[0],
                seasonal_order_range[1] + 1
            )

            ar_coefs = np.random.uniform(coef_range[0], coef_range[1], ar_order)
            ma_coefs = np.random.uniform(coef_range[0], coef_range[1], ma_order)

            seasonal_ar_coefs = np.random.uniform(
                seasonal_coef_range[0],
                seasonal_coef_range[1],
                seasonal_ar_order
            )

            seasonal_ma_coefs = np.random.uniform(
                seasonal_coef_range[0],
                seasonal_coef_range[1],
                seasonal_ma_order
            )

            nonseasonal_ar = np.r_[1, -ar_coefs]
            nonseasonal_ma = np.r_[1, ma_coefs]

            seasonal_ar = np.zeros(seasonal_ar_order * s + 1)
            seasonal_ar[0] = 1
            for i, coef in enumerate(seasonal_ar_coefs, start=1):
                seasonal_ar[i * s] = -coef

            seasonal_ma = np.zeros(seasonal_ma_order * s + 1)
            seasonal_ma[0] = 1
            for i, coef in enumerate(seasonal_ma_coefs, start=1):
                seasonal_ma[i * s] = coef

            ar_poly = np.convolve(nonseasonal_ar, seasonal_ar)
            ma_poly = np.convolve(nonseasonal_ma, seasonal_ma)

            arma_process = ArmaProcess(ar_poly, ma_poly)

            if arma_process.isstationary and arma_process.isinvertible:
                burnin = max(200, 8 * s)

                sarma_error = arma_process.generate_sample(
                    nsample=n,
                    burnin=burnin,
                    scale=1.0
                )

                error_std = np.std(sarma_error)
                if error_std > 1e-8:
                    sarma_error = (sarma_error - np.mean(sarma_error)) / error_std

                break

        if sarma_error is None:
            raise RuntimeError("Could not generate valid stationary SARMA error.")

        if error_scale is None:
            error_scale = amplitude * np.random.uniform(0.25, 0.60)

        sarma_error = sarma_error * error_scale

        noise_std = noise_std if noise_std is not None else np.random.uniform(0.01, 0.05)
        observation_noise = np.random.normal(0, noise_std, size=n)

        series = deterministic_seasonality + sarma_error + observation_noise

        df = pd.DataFrame({
            "time": np.arange(n),
            "data": series,
            "stationary": np.zeros(n).astype(int),
            "seasonal": np.ones(n).astype(int),
            "sarma": np.ones(n).astype(int)
        })

        info = {
            "type": "seasonal",
            "subtype": "SARMA",
            "periods": [s],
            "period_meanings": {
                s: self.get_period_meanings(s)
            },
            "diff": 0,
            "seasonal_diff": 0,
            "unit_root": "none",
            "amplitude": amplitude,
            "noise_std": noise_std,
            "scale_factor": scale_factor,
            "num_harmonics": num_harmonics,
            "fourier_coefficients": fourier_coefficients,
            "ar_order": ar_order,
            "ma_order": ma_order,
            "seasonal_ar_order": seasonal_ar_order,
            "seasonal_ma_order": seasonal_ma_order,
            "ar_coefs": ar_coefs,
            "ma_coefs": ma_coefs,
            "seasonal_ar_coefs": seasonal_ar_coefs,
            "seasonal_ma_coefs": seasonal_ma_coefs
        }

        return df, info

    #VOLATILITY

    def generate_volatility(self, kind = None):
        if kind == 'arch':
            series, info = self.generate_arch_series(self.length)
        elif kind == 'garch':
            series, info = self.generate_garch_series(self.length)
        elif kind == 'egarch':
            series, info = self.generate_egarch_series(self.length)
        elif kind == 'aparch':
            series, info = self.generate_aparch_series(self.length)

        df = pd.DataFrame({
            'time': np.arange(self.length),
            'data': series,
            'stationary': (np.zeros(self.length)).astype(int),
            'seasonal': np.zeros(self.length).astype(int),
        })
        return df, info


    def generate_seasonality_from_base_series(
        self,
        kind=None,
        num_components=2,
        period=None
    ):
        """
        Generates seasonal base series from scratch.

        kind:
            "single"   -> deterministic single Fourier seasonality
            "multiple" -> deterministic multiple Fourier seasonality
            "sarima"   -> seasonal unit root:
                          Y_t - Y_{t-s} = Fourier(t) + noise
            "sarma"    -> deterministic Fourier seasonality with SARMA errors
        """

        if kind is None:
            kind = random.choice(["single", "multiple", "sarma", "sarima"])

        if kind == "single":
            df, info = self.generate_single_seasonality(
                period=period,
                num_harmonics=1
            )

        elif kind == "multiple":
            periods = self.normalize_period_list(period)

            df, info = self.generate_multiple_seasonality(
                num_components=num_components,
                periods=periods,
                num_harmonics=1
            )

        elif kind == "sarima":
            df, info = self.generate_sarima_series(
                period=period,
                num_harmonics=1
            )

        elif kind == "sarma":
            df, info = self.generate_deterministic_sarma_series(
                period=period,
                num_harmonics=1
            )

        else:
            raise ValueError(
                "Invalid kind. Choose from 'single', 'multiple', 'sarma' or 'sarima'."
            )

        return df, info

    #STRUCTURAL BREAKS
    
    def generate_mean_shift(self, df, num_breaks=1, scale_factor=1, signs=None, location=None, 
                            noise_std=None, seasonal_period=None, slope=None, intercept=None, is_loc=None):
        series = df['data'].copy()
        n = len(series)
        noise_std = noise_std if noise_std is not None else np.random.uniform(0.01, 0.05)
        min_distance = 0.1 * n
        created_breaks = []
        magnitudes = []
        info = []
        
        if seasonal_period is None:
            seasonal_component = np.zeros(n)
            shift_target = series.copy()

        elif isinstance(seasonal_period, int):
            stl = STL(series, period=seasonal_period, robust=True)
            result = stl.fit()

            seasonal_component = result.seasonal
            shift_target = series - seasonal_component

        elif isinstance(seasonal_period, (list, tuple)):
            mstl = MSTL(series, periods=seasonal_period)
            result = mstl.fit()

            seasonal_component = result.seasonal
            shift_target = series - seasonal_component.sum(axis=1)

        else:
            raise ValueError("seasonal_period must be None, an int, or a list/tuple of ints.")
        # Decide break points
        if num_breaks == 1 and location in ["beginning", "middle", "end"]:
            if location == "beginning":
                break_points = [np.random.randint(int(0.1 * n), int(0.3 * n))]
            elif location == "middle":
                break_points = [np.random.randint(int(0.4 * n), int(0.6 * n))]
            elif location == "end":
                break_points = [np.random.randint(int(0.7 * n), int(0.9 * n))]
        else:
            candidates = np.arange(int(0.1 * n), int(0.9 * n))
            break_points = []
            while len(break_points) < num_breaks and len(candidates) > 0:
                point = np.random.choice(candidates)
                if isinstance(seasonal_period, int):
                    phase = point % seasonal_period
                    point -= phase
                elif isinstance(seasonal_period, (list, tuple)):
                    sp = np.random.choice(seasonal_period)
                    phase = point % sp
                    point -= phase
                if point not in break_points:
                    break_points.append(point)
                    candidates = candidates[np.abs(candidates - point) >= min_distance]
            break_points = sorted(break_points)

        if signs is None or len(signs) != len(break_points):
            raise ValueError("signs must be a list with the same length as the number of breaks.")

        info = {'type': 'structural_break', 'subtype': 'mean_shift', 'num_breaks':num_breaks, 'location' : location}
        
        prev_point = 0
        # Apply shifts
        for i, break_point in enumerate(break_points):
            local_std = np.std(shift_target[prev_point:break_point])
            magnitude = np.random.uniform(1.5, 3) * local_std
            magnitudes.append(magnitude)
            level_shift = signs[i] * magnitude
            shift_target[break_point:] += level_shift * scale_factor 
            created_breaks.append(break_point)
            prev_point = break_point

        info['shift_indices'] = created_breaks
        info['shift_magnitudes'] = magnitudes
    
        # Reconstruct series
        if seasonal_period is None:
            series = shift_target
        elif isinstance(seasonal_period, int):
            series = shift_target + seasonal_component
        elif isinstance(seasonal_period, (list, tuple)):
            series = shift_target + seasonal_component.sum(axis=1)

        noise = np.random.normal(0, noise_std, n)
        series += noise

        df.loc[:,'data'] = series
        df.loc[:,'stationary'] = 0

        if is_loc is True:
            mean_shift_label = np.zeros(n, dtype=int)

            for regime_number, break_point in enumerate(
                sorted(created_breaks),
                start=1
            ):
                mean_shift_label[break_point:] = regime_number

            df.loc[:, "mean_shift_label"] = mean_shift_label

        return df, info

    def generate_variance_shift(
        self,
        df,
        num_breaks=1,
        scale_factor=1,
        signs=None,
        location=None,
        seasonal_period=None,
        slope=None,
        intercept=None,
        is_loc=None
    ):
        series = df["data"].copy()
        n = len(series)

        min_distance = 0.1 * n
        created_breaks = []
        variance_change_factors = []

        if seasonal_period is None:
            seasonal_component = np.zeros(n)

            if slope is not None and intercept is not None:
                trend_component = intercept + slope * np.arange(n)
                residual_component = series - trend_component
            else:
                trend_component = np.zeros(n)
                residual_component = series.copy()

        elif isinstance(seasonal_period, int):
            stl = STL(
                series,
                period=seasonal_period,
                robust=True
            )
            result = stl.fit()

            trend_component = result.trend
            seasonal_component = result.seasonal
            residual_component = result.resid

        elif isinstance(seasonal_period, (list, tuple)):
            mstl = MSTL(
                series,
                periods=seasonal_period
            )
            result = mstl.fit()

            trend_component = result.trend
            seasonal_component = result.seasonal
            residual_component = result.resid

        else:
            raise ValueError(
                "seasonal_period must be None, "
                "an int, or a list/tuple of ints."
            )

        # Decide break points
        if (
            num_breaks == 1
            and location in ["beginning", "middle", "end"]
        ):
            if location == "beginning":
                break_points = [
                    np.random.randint(
                        int(0.1 * n),
                        int(0.3 * n)
                    )
                ]

            elif location == "middle":
                break_points = [
                    np.random.randint(
                        int(0.4 * n),
                        int(0.6 * n)
                    )
                ]

            elif location == "end":
                break_points = [
                    np.random.randint(
                        int(0.7 * n),
                        int(0.9 * n)
                    )
                ]

        else:
            candidates = np.arange(
                int(0.1 * n),
                int(0.9 * n)
            )

            break_points = []

            while (
                len(break_points) < num_breaks
                and len(candidates) > 0
            ):
                point = np.random.choice(candidates)

                if isinstance(seasonal_period, int):
                    phase = point % seasonal_period
                    point -= phase

                elif isinstance(
                    seasonal_period,
                    (list, tuple)
                ):
                    sp = np.random.choice(seasonal_period)
                    phase = point % sp
                    point -= phase

                if point not in break_points:
                    break_points.append(point)

                    candidates = candidates[
                        np.abs(candidates - point)
                        >= min_distance
                    ]

            break_points = sorted(break_points)

        if signs is None or len(signs) != len(break_points):
            raise ValueError(
                "signs must be a list with the same "
                "length as the number of breaks."
            )

        info = {
            "type": "structural_break",
            "subtype": "variance_shift",
            "num_breaks": len(break_points),
            "location": location
        }

        # Apply variance shifts
        for i, break_point in enumerate(break_points):
            variance_factor = np.random.uniform(1.5, 3)
            variance_change_factors.append(variance_factor)

            if signs[i] > 0:
                residual_component[break_point:] *= (
                    variance_factor * scale_factor
                )

            elif signs[i] < 0:
                residual_component[break_point:] /= (
                    variance_factor * scale_factor
                )

            created_breaks.append(break_point)

        # Reconstruct series
        if seasonal_period is None:
            series = (
                trend_component
                + residual_component
            )

        elif isinstance(seasonal_period, int):
            series = (
                trend_component
                + seasonal_component
                + residual_component
            )

        elif isinstance(seasonal_period, (list, tuple)):
            series = (
                trend_component
                + seasonal_component.sum(axis=1)
                + residual_component
            )

        info["shift_indices"] = created_breaks
        info["shift_magnitudes"] = variance_change_factors

        df.loc[:, "data"] = series
        df.loc[:, "stationary"] = 0

        # Create structural-break regime labels only when requested
        if is_loc is True:
            variance_shift_label = np.zeros(
                n,
                dtype=int
            )

            for regime_number, break_point in enumerate(
                sorted(created_breaks),
                start=1
            ):
                variance_shift_label[
                    break_point:
                ] = regime_number

            df.loc[
                :,
                "variance_shift_label"
            ] = variance_shift_label

        return df, info

    def generate_trend_shift(
        self,
        df,
        location="middle",
        num_breaks=1,
        scale_factor=1,
        change_types=None,
        slope=None,
        intercept=None,
        seasonal_period=None,
        noise_std=None,
        is_loc=None
    ):
        series = df["data"].copy()
        n = len(series)

        min_distance = 0.1 * n

        noise_std = (
            noise_std
            if noise_std is not None
            else np.random.uniform(0.01, 0.05)
        )

        created_breaks = []
        created_change_types = []

        if slope is None or intercept is None:
            raise ValueError(
                "slope and intercept must be provided for trend shift."
            )

        if seasonal_period is None:
            original_trend = intercept + slope * np.arange(n)
            residual_component = series - original_trend

        elif isinstance(seasonal_period, int):
            stl = STL(
                series,
                period=seasonal_period,
                robust=True
            )
            result = stl.fit()

            seasonal_component = result.seasonal
            residual_component = result.resid

        elif isinstance(seasonal_period, (list, tuple)):
            mstl = MSTL(
                series,
                periods=seasonal_period
            )
            result = mstl.fit()

            seasonal_component = result.seasonal
            residual_component = result.resid

        else:
            raise ValueError(
                "seasonal_period must be None, "
                "an int, or a list/tuple of ints."
            )

        # Decide break points
        if (
            num_breaks == 1
            and location in ["beginning", "middle", "end"]
        ):
            if location == "beginning":
                break_points = [
                    np.random.randint(
                        int(0.1 * n),
                        int(0.3 * n)
                    )
                ]

            elif location == "middle":
                break_points = [
                    np.random.randint(
                        int(0.4 * n),
                        int(0.6 * n)
                    )
                ]

            elif location == "end":
                break_points = [
                    np.random.randint(
                        int(0.7 * n),
                        int(0.9 * n)
                    )
                ]

        else:
            candidates = np.arange(
                int(0.1 * n),
                int(0.9 * n)
            )

            break_points = []

            while (
                len(break_points) < num_breaks
                and len(candidates) > 0
            ):
                point = np.random.choice(candidates)

                if isinstance(seasonal_period, int):
                    phase = point % seasonal_period
                    point -= phase

                elif isinstance(seasonal_period, (list, tuple)):
                    sp = np.random.choice(seasonal_period)
                    phase = point % sp
                    point -= phase

                if point not in break_points:
                    break_points.append(point)

                    candidates = candidates[
                        np.abs(candidates - point)
                        >= min_distance
                    ]

            break_points = sorted(break_points)

        # Validate change_types input
        if (
            change_types is None
            or len(change_types) != len(break_points)
        ):
            raise ValueError(
                "change_types must be a list with the same "
                "length as the number of breaks."
            )

        # Initialize trend array
        current_slope = slope
        current_level = intercept

        trend = np.zeros(n)
        prev_point = 0

        info = {
            "type": "structural_break",
            "subtype": "trend_shift",
            "num_breaks": len(break_points),
            "location": location
        }

        # Construct piecewise trend
        for i, break_point in enumerate(break_points + [n]):
            slope_change_factor = np.random.uniform(1.5, 4.5)
            segment_length = break_point - prev_point

            if segment_length > 0:
                segment_trend = (
                    current_level
                    + current_slope * np.arange(segment_length)
                )

                trend[prev_point:break_point] = segment_trend
                current_level = segment_trend[-1]

            if break_point == n:
                break

            change_type = change_types[i]

            if change_type == "direction_change":
                current_slope = -current_slope

            elif change_type == "magnitude_change":
                current_slope = (
                    current_slope
                    * slope_change_factor
                    * scale_factor
                )

            elif change_type == "direction_and_magnitude_change":
                current_slope = (
                    -current_slope
                    * slope_change_factor
                    * scale_factor
                )

            else:
                raise ValueError(
                    "Invalid change_type: "
                    + str(change_type)
                )

            created_breaks.append(break_point)
            created_change_types.append(change_type)
            prev_point = break_point

        info["shift_indices"] = created_breaks
        info["shift_types"] = created_change_types

        # Reconstruct series
        noise = np.random.normal(
            0,
            noise_std,
            size=n
        )

        if seasonal_period is None:
            series = (
                trend
                + residual_component
                + noise
            )

        elif isinstance(seasonal_period, int):
            series = (
                trend
                + seasonal_component
                + residual_component
                + noise
            )

        elif isinstance(seasonal_period, (list, tuple)):
            series = (
                trend
                + seasonal_component.sum(axis=1)
                + residual_component
                + noise
            )

        # Update dataframe
        df.loc[:, "data"] = series
        df.loc[:, "stationary"] = 0

        # Create structural-break regime labels only when requested
        if is_loc is True:
            trend_shift_label = np.zeros(
                n,
                dtype=int
            )

            for regime_number, break_point in enumerate(
                sorted(created_breaks),
                start=1
            ):
                trend_shift_label[break_point:] = regime_number

            df.loc[:, "trend_shift_label"] = trend_shift_label

        return df, info

In [4]:

# =========================
# Utility helpers
# =========================

def run_quietly(func, *args, quiet=True, **kwargs):
    """Runs a function while suppressing stdout/stderr."""
    if not quiet:
        return func(*args, **kwargs)

    with io.StringIO() as out_buffer, io.StringIO() as err_buffer:
        with redirect_stdout(out_buffer), redirect_stderr(err_buffer):
            return func(*args, **kwargs)


def make_json_safe(obj):
    """Converts numpy/pandas objects into JSON-safe Python objects."""
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if pd.isna(obj) if isinstance(obj, (float, np.floating)) else False:
        return None
    return obj


def parse_period_list(value):
    """Safely parses true/detected periods from lists, strings, JSON, or NaN."""
    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    if isinstance(value, (list, tuple, set, np.ndarray)):
        result = []
        for item in list(value):
            result.extend(parse_period_list(item))
        return [float(v) for v in result]

    if isinstance(value, (int, float, np.integer, np.floating)):
        if np.isfinite(value):
            return [float(value)]
        return []

    if isinstance(value, str):
        value = value.strip()
        if value == "" or value.lower() in ["nan", "none", "null"]:
            return []
        try:
            return parse_period_list(json.loads(value))
        except Exception:
            pass
        try:
            return parse_period_list(ast.literal_eval(value))
        except Exception:
            return []

    return []


def extract_true_periods(info):
    """Extracts periods from the info dictionary. Used only for evaluation, not detection."""
    if not isinstance(info, dict):
        return []
    for key in ["periods", "period", "seasonal_periods", "seasonal_period"]:
        if key in info:
            return parse_period_list(info[key])
    return []


def clean_series(series):
    x = np.asarray(series, dtype=float)
    return x[np.isfinite(x)]


def safe_divide(numerator, denominator):
    if denominator == 0 or pd.isna(denominator):
        return np.nan
    return numerator / denominator


def periods_match(p1, p2, absolute_tolerance=1.0, relative_tolerance=0.10):
    p1 = float(p1)
    p2 = float(p2)
    tolerance = max(absolute_tolerance, relative_tolerance * abs(p1))
    return abs(p1 - p2) <= tolerance


In [5]:

# =========================
# Candidate period helpers
# =========================

SAMPLING_FREQUENCY_PERIODS = {
    "monthly": [3, 6, 12],
    "quarterly": [4],
    "daily": [7, 30, 90, 180, 365],
    "weekly": [4, 13, 26, 52],
    "business_daily": [5, 21, 63, 126, 252],
    "hourly": [24, 168],
}


def get_global_calendar_periods():
    """Uses the generator's global calendar period list."""
    ts = TimeSeriesGenerator(length=LENGTH)
    return sorted(set(int(p) for p in ts.get_all_calendar_periods()))


def get_valid_candidate_periods(n, allowed_periods=None, min_cycles=6):
    """
    Returns valid candidate periods for a series length.

    This does not use the true periods from info. It only defines the search space.
    """
    if allowed_periods is None:
        allowed_periods = get_global_calendar_periods()

    periods = sorted(set(int(p) for p in allowed_periods))
    max_period = int(n // min_cycles)

    return [p for p in periods if 2 <= p <= max_period]


def get_candidates_for_real_series(series, sampling_frequency=None, custom_candidate_periods=None, min_cycles=6):
    """
    Candidate-period function for real data later.

    - If custom_candidate_periods is given, it uses those.
    - Else if sampling_frequency is given, it uses common periods for that frequency.
    - Else it uses the global calendar-period list.
    """
    n = len(clean_series(series))

    if custom_candidate_periods is not None:
        allowed = custom_candidate_periods
    elif sampling_frequency is not None:
        if sampling_frequency not in SAMPLING_FREQUENCY_PERIODS:
            raise ValueError(f"Unknown sampling_frequency={sampling_frequency}. Valid keys: {list(SAMPLING_FREQUENCY_PERIODS)}")
        allowed = SAMPLING_FREQUENCY_PERIODS[sampling_frequency]
    else:
        allowed = get_global_calendar_periods()

    return get_valid_candidate_periods(n, allowed_periods=allowed, min_cycles=min_cycles)


In [6]:

# =========================
# Detection features
# =========================

def preprocess_for_spectrum(series, detrend_method="constant"):
    x = clean_series(series)

    if detrend_method == "constant":
        return x - np.mean(x)
    if detrend_method == "linear":
        return scipy_detrend(x, type="linear")
    if detrend_method is None:
        return x.copy()

    raise ValueError("detrend_method must be 'constant', 'linear', or None.")


def compute_full_periodogram(series, detrend_method="constant"):
    """Computes Fourier periodogram for plotting/diagnostics."""
    x = clean_series(series)
    n = len(x)
    y = preprocess_for_spectrum(x, detrend_method=detrend_method)

    fft_values = np.fft.rfft(y)
    k = np.arange(len(fft_values))
    power = (2.0 / n) * np.abs(fft_values) ** 2

    if len(power) > 0:
        power[0] = 0.0
    if n % 2 == 0 and len(power) > 1:
        power[-1] = 0.0

    period = np.full_like(k, fill_value=np.inf, dtype=float)
    mask = k > 0
    period[mask] = n / k[mask]

    return pd.DataFrame({
        "k": k,
        "period": period,
        "power": power,
        "power_share": power / (np.sum(power) + 1e-12)
    })


def periodogram_power_share_for_period(series, period, detrend_method="constant", window=1):
    """Power share near a candidate period."""
    x = clean_series(series)
    n = len(x)
    y = preprocess_for_spectrum(x, detrend_method=detrend_method)

    fft_values = np.fft.rfft(y)
    power = (2.0 / n) * np.abs(fft_values) ** 2

    if len(power) > 0:
        power[0] = 0.0
    if n % 2 == 0 and len(power) > 1:
        power[-1] = 0.0

    total_power = np.sum(power)
    if total_power <= 0:
        return 0.0

    target_k = int(round(n / period))
    if target_k <= 0 or target_k >= len(power):
        return 0.0

    low = max(1, target_k - window)
    high = min(len(power) - 1, target_k + window)
    local_power = np.max(power[low:high + 1])

    return float(local_power / total_power)


def autocorrelation_at_lag(series, lag):
    x = clean_series(series)
    lag = int(lag)

    if lag <= 0 or lag >= len(x):
        return np.nan

    x = x - np.mean(x)
    denominator = np.sum(x ** 2)
    if denominator <= 0:
        return np.nan

    numerator = np.sum(x[:-lag] * x[lag:])
    return float(numerator / denominator)


def stl_seasonal_strength(series, period):
    """
    Seasonal strength from STL:
        strength = max(0, 1 - Var(resid) / Var(resid + seasonal))
    """
    x = clean_series(series)
    period = int(period)

    if len(x) < 2 * period or period < 2:
        return np.nan

    try:
        result = STL(x, period=period, robust=True).fit()
        seasonal = result.seasonal
        resid = result.resid
        denominator = np.var(resid + seasonal)
        if denominator <= 0:
            return 0.0
        return float(max(0.0, 1.0 - np.var(resid) / denominator))
    except Exception:
        return np.nan


def make_fourier_features(n, period, num_harmonics=1):
    t = np.arange(n)
    features = []
    for h in range(1, num_harmonics + 1):
        features.append(np.sin(2 * np.pi * h * t / period))
        features.append(np.cos(2 * np.pi * h * t / period))
    return np.column_stack(features)


def fourier_regression_improvement(series, period, num_harmonics=1, include_trend=True):
    """
    Compares baseline model vs baseline + Fourier terms.
    Positive BIC improvement means Fourier terms improved the model enough.
    """
    x = clean_series(series)
    n = len(x)
    y = x.copy()
    t = np.arange(n)

    if include_trend:
        t_scaled = (t - np.mean(t)) / (np.std(t) + 1e-12)
        X_base = np.column_stack([np.ones(n), t_scaled])
    else:
        X_base = np.ones((n, 1))

    X_fourier = make_fourier_features(n, period=period, num_harmonics=num_harmonics)
    X_full = np.column_stack([X_base, X_fourier])

    base_model = LinearRegression(fit_intercept=False).fit(X_base, y)
    full_model = LinearRegression(fit_intercept=False).fit(X_full, y)

    y_base = base_model.predict(X_base)
    y_full = full_model.predict(X_full)

    mse_base = mean_squared_error(y, y_base)
    mse_full = mean_squared_error(y, y_full)
    mse_improvement = 0.0 if mse_base <= 0 else 1.0 - (mse_full / mse_base)

    rss_base = np.sum((y - y_base) ** 2)
    rss_full = np.sum((y - y_full) ** 2)

    k_base = X_base.shape[1]
    k_full = X_full.shape[1]
    eps = 1e-12

    bic_base = n * np.log((rss_base + eps) / n) + k_base * np.log(n)
    bic_full = n * np.log((rss_full + eps) / n) + k_full * np.log(n)

    return {
        "mse_improvement": float(mse_improvement),
        "bic_improvement": float(bic_base - bic_full)
    }


In [7]:

# =========================
# Conservative period detector
# =========================

def conservative_period_detection(
    series,
    candidate_periods,
    max_detected_periods=5,
    detrend_method="constant",
    periodogram_power_threshold=0.06,
    stl_strength_threshold=0.20,
    bic_improvement_threshold=10.0,
    min_pass_count=3,
    top_candidates_for_expensive_tests=8,
    num_harmonics=1,
):
    """
    Conservative period detector.

    It does NOT use true periods. It tests candidate periods and accepts a period
    only when multiple evidence sources support it.

    Fast design:
    - Periodogram and ACF are computed for all candidate periods.
    - STL and Fourier regression are computed only for the top candidates.
    """
    x = clean_series(series)
    n = len(x)

    acf_threshold = 2.0 / np.sqrt(n)

    base_rows = []

    for period in candidate_periods:
        pgram_share = periodogram_power_share_for_period(
            x,
            period=period,
            detrend_method=detrend_method,
            window=1
        )
        acf_value = autocorrelation_at_lag(x, period)
        acf_abs = 0.0 if not np.isfinite(acf_value) else abs(acf_value)

        preliminary_score = (2.0 * pgram_share) + acf_abs

        base_rows.append({
            "period": int(period),
            "periodogram_power_share": float(pgram_share),
            "acf": float(acf_value) if np.isfinite(acf_value) else np.nan,
            "acf_abs": float(acf_abs),
            "acf_threshold": float(acf_threshold),
            "preliminary_score": float(preliminary_score),
        })

    scores = pd.DataFrame(base_rows)

    if scores.empty:
        return [], scores

    scores = scores.sort_values("preliminary_score", ascending=False).reset_index(drop=True)
    top_periods = scores.head(top_candidates_for_expensive_tests)["period"].tolist()

    # Initialize expensive columns for all candidates.
    scores["stl_strength"] = np.nan
    scores["mse_improvement"] = np.nan
    scores["bic_improvement"] = np.nan

    for period in top_periods:
        idx = scores.index[scores["period"] == period][0]

        stl_strength = stl_seasonal_strength(x, period)
        fourier_info = fourier_regression_improvement(
            x,
            period=period,
            num_harmonics=num_harmonics,
            include_trend=True
        )

        scores.loc[idx, "stl_strength"] = stl_strength
        scores.loc[idx, "mse_improvement"] = fourier_info["mse_improvement"]
        scores.loc[idx, "bic_improvement"] = fourier_info["bic_improvement"]

    scores["pass_periodogram"] = scores["periodogram_power_share"] >= periodogram_power_threshold
    scores["pass_acf"] = scores["acf_abs"] >= scores["acf_threshold"]
    scores["pass_stl"] = scores["stl_strength"] >= stl_strength_threshold
    scores["pass_fourier"] = scores["bic_improvement"] >= bic_improvement_threshold

    scores["pass_count"] = (
        scores["pass_periodogram"].astype(int)
        + scores["pass_acf"].astype(int)
        + scores["pass_stl"].fillna(False).astype(int)
        + scores["pass_fourier"].fillna(False).astype(int)
    )

    scores["final_score"] = (
        2.0 * scores["periodogram_power_share"].fillna(0)
        + 1.0 * scores["acf_abs"].fillna(0)
        + 1.5 * scores["stl_strength"].fillna(0)
        + 0.05 * scores["bic_improvement"].fillna(0).clip(lower=0)
    )

    scores["accepted"] = scores["pass_count"] >= min_pass_count

    accepted = scores[scores["accepted"]].copy()
    accepted = accepted.sort_values("final_score", ascending=False)

    detected_periods = accepted["period"].head(max_detected_periods).astype(int).tolist()

    scores = scores.sort_values("final_score", ascending=False).reset_index(drop=True)

    return detected_periods, scores

def get_candidate_score(candidate_scores, period):
    """
    Returns the score of a candidate period from candidate_scores.
    If score column does not exist, falls back to available evidence columns.
    """

    if candidate_scores is None or len(candidate_scores) == 0:
        return 0.0

    rows = candidate_scores[
        candidate_scores["period"].astype(int) == int(period)
    ]

    if len(rows) == 0:
        return 0.0

    row = rows.iloc[0]

    if "score" in candidate_scores.columns:
        return float(row["score"])

    score = 0.0

    for col in [
        "periodogram_power_share",
        "stl_strength",
        "mse_improvement",
        "bic_improvement"
    ]:
        if col in candidate_scores.columns:
            value = row[col]
            if pd.notna(value):
                score += float(value)

    return score


def are_harmonically_related(p1, p2, tolerance=0.08):
    """
    Checks whether two periods are approximately integer multiples.

    Example:
        12 and 6  -> related
        24 and 12 -> related
        52 and 26 -> related
        30 and 7  -> not related
    """

    p1 = float(p1)
    p2 = float(p2)

    small = min(p1, p2)
    large = max(p1, p2)

    if small <= 0:
        return False

    ratio = large / small
    nearest_integer = round(ratio)

    if nearest_integer < 2:
        return False

    relative_error = abs(ratio - nearest_integer) / nearest_integer

    return relative_error <= tolerance


def prune_detected_periods(
    detected_periods,
    candidate_scores,
    harmonic_tolerance=0.08,
    min_relative_score_to_best=0.55,
    max_periods=3
):
    """
    Removes extra detected periods that are probably harmonics/subharmonics
    or weak secondary detections.

    This does NOT use true periods.
    """

    detected_periods = [
        int(round(float(p)))
        for p in parse_period_list(detected_periods)
    ]

    if len(detected_periods) <= 1:
        return detected_periods

    scored_periods = []

    for period in detected_periods:
        score = get_candidate_score(candidate_scores, period)
        scored_periods.append((period, score))

    scored_periods = sorted(
        scored_periods,
        key=lambda item: item[1],
        reverse=True
    )

    best_score = scored_periods[0][1]

    kept = []

    for period, score in scored_periods:

        if best_score > 0:
            relative_score = score / best_score
        else:
            relative_score = 1.0

        if relative_score < min_relative_score_to_best:
            continue

        harmonically_redundant = any(
            are_harmonically_related(period, kept_period, tolerance=harmonic_tolerance)
            for kept_period in kept
        )

        if harmonically_redundant:
            continue

        kept.append(period)

        if len(kept) >= max_periods:
            break

    return sorted(kept)

def detect_periods_for_real_series(
    series,
    sampling_frequency=None,
    custom_candidate_periods=None,
    min_cycles=6,
):
    """
    Convenience wrapper for real data later.
    No true-period info is required.
    """
    candidates = get_candidates_for_real_series(
        series,
        sampling_frequency=sampling_frequency,
        custom_candidate_periods=custom_candidate_periods,
        min_cycles=min_cycles
    )

    detected, scores = conservative_period_detection(
        series=series,
        candidate_periods=candidates,
        max_detected_periods=MAX_DETECTED_PERIODS,
        detrend_method=DETREND_METHOD,
        periodogram_power_threshold=PERIODOGRAM_POWER_THRESHOLD,
        stl_strength_threshold=STL_STRENGTH_THRESHOLD,
        bic_improvement_threshold=BIC_IMPROVEMENT_THRESHOLD,
        min_pass_count=MIN_PASS_COUNT,
        top_candidates_for_expensive_tests=TOP_CANDIDATES_FOR_EXPENSIVE_TESTS,
    )

    return detected, scores


In [8]:
# =========================
# Period-level evaluation
# =========================

def normalize_period_set(periods):
    """
    Converts a period list into a clean integer set.

    IMPORTANT:
    This does exact candidate-period evaluation.
    No tolerance is used here, because otherwise true period 4 also makes
    candidate periods 3 and 5 look true when tolerance is ±1.
    """
    periods = parse_period_list(periods)
    clean_periods = set()

    for p in periods:
        if p is None:
            continue
        if np.isfinite(p):
            clean_periods.add(int(round(float(p))))

    return clean_periods


def period_level_confusion_counts(
    true_periods,
    detected_periods,
    candidate_periods,
    absolute_tolerance=None,
    relative_tolerance=None,
):
    """
    Computes TP/TN/FP/FN at candidate-period level using EXACT membership.

    Each candidate period is one item.

    Example 1:
        true=[4], detected=[4], candidates=[3,4,5]
        -> TP=1, TN=2, FP=0, FN=0

    Example 2:
        true=[7, 30, 52], detected=[7, 30]
        -> TP=2, FN=1, FP=0, TN=remaining candidates

    Notes
    -----
    absolute_tolerance and relative_tolerance are kept only so the function
    call in run_experiment does not need to change. They are intentionally
    NOT used in this exact candidate-level evaluation.
    """
    true_set = normalize_period_set(true_periods)
    detected_set = normalize_period_set(detected_periods)
    candidate_set = set(int(round(float(p))) for p in candidate_periods)

    true_set = true_set.intersection(candidate_set)
    detected_set = detected_set.intersection(candidate_set)

    rows = []

    for candidate in sorted(candidate_set):
        is_true = candidate in true_set
        is_detected = candidate in detected_set

        if is_true and is_detected:
            label = "TP"
        elif (not is_true) and (not is_detected):
            label = "TN"
        elif (not is_true) and is_detected:
            label = "FP"
        elif is_true and (not is_detected):
            label = "FN"
        else:
            label = "UNKNOWN"

        rows.append({
            "candidate_period": candidate,
            "is_true_period": bool(is_true),
            "is_detected_period": bool(is_detected),
            "confusion_label": label,
        })

    item_df = pd.DataFrame(rows)
    counts = item_df["confusion_label"].value_counts().to_dict()

    tp = int(counts.get("TP", 0))
    tn = int(counts.get("TN", 0))
    fp = int(counts.get("FP", 0))
    fn = int(counts.get("FN", 0))

    is_seasonal = len(true_set) > 0
    success_any = np.nan
    success_all = np.nan

    if is_seasonal:
        success_any = tp > 0
        success_all = fn == 0

    return {
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "success_any": success_any,
        "success_all": success_all,
        "period_item_count": len(candidate_set),
        "period_item_details": item_df,
    }

In [9]:
def safe_divide(numerator, denominator):
    if denominator == 0:
        return np.nan
    return numerator / denominator


def build_summary_tables(results_df):
    """
    Builds summary tables from per-series TP/TN/FP/FN counts.

    results_df must already contain:
        TP, TN, FP, FN, success_any, success_all, example_type
    """

    df = results_df.copy()

    summary_by_type = (
        df
        .groupby("example_type")
        .agg(
            n_series=("series_id", "count"),
            true_period_count=("true_period_count", "sum"),
            detected_period_count=("detected_period_count", "sum"),
            period_item_count=("period_item_count", "sum"),
            TP=("TP", "sum"),
            TN=("TN", "sum"),
            FP=("FP", "sum"),
            FN=("FN", "sum"),
            success_any_rate=("success_any", "mean"),
            success_all_rate=("success_all", "mean"),
        )
        .reset_index()
    )

    summary_by_type["TP_rate"] = summary_by_type.apply(
        lambda row: safe_divide(row["TP"], row["TP"] + row["FN"]),
        axis=1
    )

    summary_by_type["TN_rate"] = summary_by_type.apply(
        lambda row: safe_divide(row["TN"], row["TN"] + row["FP"]),
        axis=1
    )

    summary_by_type["FP_rate"] = summary_by_type.apply(
        lambda row: safe_divide(row["FP"], row["FP"] + row["TN"]),
        axis=1
    )

    summary_by_type["FN_rate"] = summary_by_type.apply(
        lambda row: safe_divide(row["FN"], row["FN"] + row["TP"]),
        axis=1
    )

    overall_summary = pd.DataFrame([{
        "example_type": "OVERALL",
        "n_series": len(df),
        "true_period_count": df["true_period_count"].sum(),
        "detected_period_count": df["detected_period_count"].sum(),
        "period_item_count": df["period_item_count"].sum(),
        "TP": df["TP"].sum(),
        "TN": df["TN"].sum(),
        "FP": df["FP"].sum(),
        "FN": df["FN"].sum(),
        "success_any_rate": df["success_any"].mean(),
        "success_all_rate": df["success_all"].mean(),
    }])

    overall_summary["TP_rate"] = overall_summary.apply(
        lambda row: safe_divide(row["TP"], row["TP"] + row["FN"]),
        axis=1
    )

    overall_summary["TN_rate"] = overall_summary.apply(
        lambda row: safe_divide(row["TN"], row["TN"] + row["FP"]),
        axis=1
    )

    overall_summary["FP_rate"] = overall_summary.apply(
        lambda row: safe_divide(row["FP"], row["FP"] + row["TN"]),
        axis=1
    )

    overall_summary["FN_rate"] = overall_summary.apply(
        lambda row: safe_divide(row["FN"], row["FN"] + row["TP"]),
        axis=1
    )

    return summary_by_type, overall_summary

In [10]:

# =========================
# Plotting helper: save only by default
# =========================

def plot_series_and_period_detection(
    series,
    true_periods,
    detected_periods,
    series_id,
    example_type,
    save_path=None,
    show_plot=False,
    detrend_method="constant",
    max_period_to_show=None,
):
    """
    Saves a two-panel plot:
      1. actual generated time series
      2. periodogram with true and detected period lines

    It does NOT show plots unless show_plot=True.
    """
    x = clean_series(series)
    n = len(x)
    t = np.arange(n)

    true_periods = parse_period_list(true_periods)
    detected_periods = parse_period_list(detected_periods)

    periodogram_df = compute_full_periodogram(x, detrend_method=detrend_method)
    plot_df = periodogram_df[np.isfinite(periodogram_df["period"]) & (periodogram_df["period"] > 1)].copy()

    if max_period_to_show is None:
        max_period_to_show = n // MIN_CYCLES

    plot_df = plot_df[plot_df["period"] <= max_period_to_show]
    plot_df = plot_df.sort_values("period")

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)

    axes[0].plot(t, x, linewidth=1)
    axes[0].set_title(
        f"{series_id} | {example_type}\nTrue periods: {true_periods} | Detected periods: {detected_periods}"
    )
    axes[0].set_xlabel("Time")
    axes[0].set_ylabel("Value")
    axes[0].grid(True, alpha=0.25)

    axes[1].plot(plot_df["period"], plot_df["power"], linewidth=1)
    axes[1].set_xlabel("Candidate period")
    axes[1].set_ylabel("Periodogram power")
    axes[1].set_title("Periodogram")
    axes[1].grid(True, alpha=0.25)

    for p in true_periods:
        if p <= max_period_to_show:
            axes[1].axvline(p, linestyle="--", linewidth=1.5, label=f"True {p:g}")

    for p in detected_periods:
        if p <= max_period_to_show:
            axes[1].axvline(p, linestyle=":", linewidth=2.0, label=f"Detected {p:g}")

    if len(true_periods) > 0 or len(detected_periods) > 0:
        axes[1].legend(loc="best")

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=160, bbox_inches="tight")

    if show_plot:
        plt.show()
    else:
        plt.close(fig)


In [11]:

# =========================
# Example generation functions
# =========================

def merge_infos(*infos, periods=None, subtype=None, type_name=None):
    """Combines info dictionaries and optionally sets true periods for evaluation."""
    merged = {
        "type": type_name if type_name is not None else "combined",
        "subtype": subtype if subtype is not None else "combined",
        "components": [make_json_safe(info) for info in infos if isinstance(info, dict)]
    }
    if periods is not None:
        merged["periods"] = parse_period_list(periods)
    return merged


def generate_example(example_type, length):
    """
    Generates one example dataframe and info dictionary.

    The true periods remain stored only in info and are used only for evaluation.
    """
    ts = TimeSeriesGenerator(length=length)

    # Seasonal examples
    if example_type == "seasonal_single":
        return ts.generate_seasonality_from_base_series(kind="single")

    if example_type == "seasonal_multiple":
        return ts.generate_seasonality_from_base_series(kind="multiple")

    if example_type == "seasonal_sarma":
        return ts.generate_seasonality_from_base_series(kind="sarma")

    if example_type == "seasonal_sarima":
        return ts.generate_seasonality_from_base_series(kind="sarima")

    # Stationary non-seasonal base examples
    if example_type.startswith("stationary_"):
        distribution = example_type.replace("stationary_", "")
        return ts.generate_stationary_base_series(distribution=distribution)

    # Stochastic trend non-seasonal examples
    if example_type.startswith("stochastic_trend_"):
        kind = example_type.replace("stochastic_trend_", "")
        return ts.generate_stochastic_trend(kind=kind)

    # Volatility non-seasonal examples
    if example_type.startswith("volatility_"):
        kind = example_type.replace("volatility_", "")
        return ts.generate_volatility(kind=kind)

    # Deterministic trend non-seasonal examples
    if example_type.startswith("deterministic_trend_"):
        base_df, base_info = ts.generate_stationary_base_series(distribution="white_noise")
        trend_kind = example_type.replace("deterministic_trend_", "")

        if trend_kind == "linear":
            df, trend_info = ts.generate_deterministic_trend_linear(base_df)
        elif trend_kind == "quadratic":
            df, trend_info = ts.generate_deterministic_trend_quadratic(base_df)
        elif trend_kind == "cubic":
            df, trend_info = ts.generate_deterministic_trend_cubic(base_df)
        elif trend_kind == "exponential":
            df, trend_info = ts.generate_deterministic_trend_exponential(base_df)
        elif trend_kind == "damped":
            df, trend_info = ts.generate_deterministic_trend_damped(base_df)
        else:
            raise ValueError(f"Unknown deterministic trend kind: {trend_kind}")

        info = merge_infos(base_info, trend_info, periods=[], subtype=example_type, type_name="nonseasonal_control")
        return df, info

    # Anomaly non-seasonal examples, except contextual_on_seasonal
    if example_type == "anomaly_single_point":
        base_df, base_info = ts.generate_stationary_base_series(distribution="white_noise")
        df, anom_info = ts.generate_point_anomaly(base_df)
        return df, merge_infos(base_info, anom_info, periods=[], subtype=example_type, type_name="nonseasonal_control")

    if example_type == "anomaly_multiple_points":
        base_df, base_info = ts.generate_stationary_base_series(distribution="white_noise")
        df, anom_info = ts.generate_point_anomalies(base_df)
        return df, merge_infos(base_info, anom_info, periods=[], subtype=example_type, type_name="nonseasonal_control")

    if example_type.startswith("anomaly_collective_"):
        shape = example_type.replace("anomaly_collective_", "")
        base_df, base_info = ts.generate_stationary_base_series(distribution="white_noise")
        df, anom_info = ts.generate_collective_anomalies(base_df, anomaly_shapes=shape)
        return df, merge_infos(base_info, anom_info, periods=[], subtype=example_type, type_name="nonseasonal_control")

    # Contextual anomaly on an actually seasonal base. This should keep the true period.
    if example_type == "anomaly_contextual_on_seasonal":
        base_df, base_info = ts.generate_single_seasonality()
        true_periods = extract_true_periods(base_info)
        seasonal_period = int(true_periods[0]) if len(true_periods) > 0 else None
        df, anom_info = ts.generate_contextual_anomalies(base_df, seasonal_period=seasonal_period)
        return df, merge_infos(base_info, anom_info, periods=true_periods, subtype=example_type, type_name="seasonal_with_anomaly")

    # Structural breaks as non-seasonal controls.
    if example_type == "structural_break_mean_shift":
        base_df, base_info = ts.generate_stationary_base_series(distribution="white_noise")
        df, break_info = ts.generate_mean_shift(base_df, num_breaks=1, signs=[random.choice([-1, 1])])
        return df, merge_infos(base_info, break_info, periods=[], subtype=example_type, type_name="nonseasonal_control")

    if example_type == "structural_break_variance_shift":
        base_df, base_info = ts.generate_stationary_base_series(distribution="white_noise")
        df, break_info = ts.generate_variance_shift(base_df, num_breaks=1, signs=[random.choice([-1, 1])])
        return df, merge_infos(base_info, break_info, periods=[], subtype=example_type, type_name="nonseasonal_control")

    if example_type == "structural_break_trend_shift":
        base_df, base_info = ts.generate_stationary_base_series(distribution="white_noise")
        df, break_info = ts.generate_trend_shift(
            base_df,
            num_breaks=1,
            slope=0.01,
            intercept=0.0,
            change_types=[random.choice(["direction_change", "magnitude_change", "direction_and_magnitude_change"])]
        )
        return df, merge_infos(base_info, break_info, periods=[], subtype=example_type, type_name="nonseasonal_control")

    raise ValueError(f"Unknown example_type: {example_type}")


EXAMPLE_TYPES = [
    # Seasonal
    "seasonal_single",
    "seasonal_multiple",
    "seasonal_sarma",
    "seasonal_sarima",

    # Stationary base controls
    "stationary_white_noise",
    "stationary_ar",
    "stationary_ma",
    "stationary_arma",

    # Stochastic trend controls
    "stochastic_trend_rw",
    "stochastic_trend_rwd",
    "stochastic_trend_ari",
    "stochastic_trend_ima",
    "stochastic_trend_arima",

    # Volatility controls
    "volatility_arch",
    "volatility_garch",
    "volatility_egarch",
    "volatility_aparch",

    # Deterministic trend controls
    "deterministic_trend_linear",
    "deterministic_trend_quadratic",
    "deterministic_trend_cubic",
    "deterministic_trend_exponential",
    "deterministic_trend_damped",

    # Anomaly controls
    "anomaly_single_point",
    "anomaly_multiple_points",
    "anomaly_collective_rectangular",
    "anomaly_collective_gaussian",
    "anomaly_collective_triangular",
    "anomaly_collective_ramp",
    "anomaly_collective_decay",
    "anomaly_contextual_on_seasonal",

    # Structural break controls
    "structural_break_mean_shift",
    "structural_break_variance_shift",
    "structural_break_trend_shift",
]

len(EXAMPLE_TYPES), EXAMPLE_TYPES[:5]


(33,
 ['seasonal_single',
  'seasonal_multiple',
  'seasonal_sarma',
  'seasonal_sarima',
  'stationary_white_noise'])

In [12]:

# =========================
# Main experiment loop
# =========================

def run_experiment(
    example_types=EXAMPLE_TYPES,
    examples_per_type=2,
    length=500,
    save_plots=True,
    show_plots=False,
    quiet_mode=True,
):
    metadata_rows = []
    result_rows = []
    failure_rows = []

    allowed_periods = get_global_calendar_periods()

    for example_type in example_types:
        for i in range(examples_per_type):
            series_id = f"{example_type}_{i:03d}"

            try:
                df, info = run_quietly(
                    generate_example,
                    example_type,
                    length,
                    quiet=quiet_mode
                )

                if df is None or info is None:
                    raise RuntimeError("Generator returned None.")

                if "data" not in df.columns:
                    raise ValueError(f"Generated dataframe has no 'data' column. Columns: {list(df.columns)}")

                series = clean_series(df["data"].values)
                true_periods = extract_true_periods(info)
                candidate_periods = get_valid_candidate_periods(
                    n=len(series),
                    allowed_periods=allowed_periods,
                    min_cycles=MIN_CYCLES,
                )

                detected_periods, candidate_scores = conservative_period_detection(
                    series=series,
                    candidate_periods=candidate_periods,
                    max_detected_periods=MAX_DETECTED_PERIODS,
                    detrend_method=DETREND_METHOD,
                    periodogram_power_threshold=PERIODOGRAM_POWER_THRESHOLD,
                    stl_strength_threshold=STL_STRENGTH_THRESHOLD,
                    bic_improvement_threshold=BIC_IMPROVEMENT_THRESHOLD,
                    min_pass_count=MIN_PASS_COUNT,
                    top_candidates_for_expensive_tests=TOP_CANDIDATES_FOR_EXPENSIVE_TESTS,
                )

                raw_detected_periods = detected_periods.copy()

                detected_periods = prune_detected_periods(
                    detected_periods=detected_periods,
                    candidate_scores=candidate_scores,
                    harmonic_tolerance=0.04,
                    min_relative_score_to_best=0.35,
                    max_periods=MAX_DETECTED_PERIODS)
                
                confusion = period_level_confusion_counts(
                    true_periods=true_periods,
                    detected_periods=detected_periods,
                    candidate_periods=candidate_periods,
                    absolute_tolerance=ABSOLUTE_PERIOD_TOLERANCE,
                    relative_tolerance=RELATIVE_PERIOD_TOLERANCE,
                )

                csv_path = CSV_DIR / f"{series_id}.csv"
                score_path = SCORE_DIR / f"{series_id}_candidate_scores.csv"
                period_items_path = SCORE_DIR / f"{series_id}_period_items.csv"
                plot_path = PLOT_DIR / f"{series_id}.png"

                df.to_csv(csv_path, index=False)
                candidate_scores.to_csv(score_path, index=False)
                confusion["period_item_details"].to_csv(period_items_path, index=False)

                if save_plots:
                    plot_series_and_period_detection(
                        series=series,
                        true_periods=true_periods,
                        detected_periods=detected_periods,
                        series_id=series_id,
                        example_type=example_type,
                        save_path=plot_path,
                        show_plot=show_plots,
                        detrend_method=DETREND_METHOD,
                    )
                else:
                    plot_path = None

                metadata_rows.append({
                    "series_id": series_id,
                    "example_type": example_type,
                    "csv_path": str(csv_path),
                    "plot_path": str(plot_path) if plot_path is not None else None,
                    "candidate_scores_path": str(score_path),
                    "period_items_path": str(period_items_path),
                    "true_periods": json.dumps(make_json_safe(true_periods)),
                    "info_json": json.dumps(make_json_safe(info)),
                })

                result_rows.append({
                    "series_id": series_id,
                    "example_type": example_type,
                    "true_periods": json.dumps(make_json_safe(true_periods)),
                    "detected_periods": json.dumps(make_json_safe(detected_periods)),
                    "true_period_count": len(true_periods),
                    "detected_period_count": len(detected_periods),
                    "candidate_periods": json.dumps(make_json_safe(candidate_periods)),
                    "period_item_count": confusion["period_item_count"],
                    "TP": confusion["TP"],
                    "TN": confusion["TN"],
                    "FP": confusion["FP"],
                    "FN": confusion["FN"],
                    "success_any": confusion["success_any"],
                    "success_all": confusion["success_all"],
                    "csv_path": str(csv_path),
                    "plot_path": str(plot_path) if plot_path is not None else None,
                    "candidate_scores_path": str(score_path),
                    "period_items_path": str(period_items_path),
                    "raw_detected_periods": json.dumps(make_json_safe(raw_detected_periods)),
                })

            except Exception as e:
                failure_rows.append({
                    "series_id": series_id,
                    "example_type": example_type,
                    "error": repr(e),
                })

    metadata_df = pd.DataFrame(metadata_rows)
    results_df = pd.DataFrame(result_rows)
    failures_df = pd.DataFrame(failure_rows)


    if len(results_df) > 0:
        summary_by_type, overall_summary = build_summary_tables(results_df)
    else:
        summary_by_type = pd.DataFrame()
        overall_summary = pd.DataFrame()

    metadata_df.to_csv(OUTPUT_DIR / "metadata.csv", index=False)
    results_df.to_csv(OUTPUT_DIR / "period_detection_results_by_series.csv", index=False)
    summary_by_type.to_csv(OUTPUT_DIR / "summary_by_example_type.csv", index=False)
    overall_summary.to_csv(OUTPUT_DIR / "overall_summary.csv", index=False)
    failures_df.to_csv(OUTPUT_DIR / "failures.csv", index=False)

    return metadata_df, results_df, summary_by_type, overall_summary, failures_df


In [13]:

# =========================
# Run experiment
# =========================

metadata_df, results_df, summary_by_type, overall_summary, failures_df = run_experiment(
    example_types=EXAMPLE_TYPES,
    examples_per_type=EXAMPLES_PER_TYPE,
    length=LENGTH,
    save_plots=SAVE_PLOTS,
    show_plots=SHOW_PLOTS,
    quiet_mode=QUIET_MODE,
)

print("Experiment finished.")
print(f"Successful series: {len(results_df)}")
print(f"Failed series: {len(failures_df)}")
print(f"Outputs saved under: {OUTPUT_DIR.resolve()}")


Experiment finished.
Successful series: 825
Failed series: 0
Outputs saved under: D:\1001TS-Stationary\cemre\betise\betise\core\period_detection_experiment_2


In [14]:

# =========================
# Clean views: no plot flood
# =========================

# Per-series view: true periods, detected periods, and period-level TP/TN/FP/FN counts.
series_view = results_df[[
    "series_id",
    "example_type",
    "true_periods",
    "detected_periods",
    "true_period_count",
    "detected_period_count",
    "TP", "TN", "FP", "FN",
    "success_any", "success_all",
    "plot_path"
]].copy()

series_view


,series_id,example_type,true_periods,detected_periods,true_period_count,detected_period_count,TP,TN,FP,FN,success_any,success_all,plot_path
0,seasonal_single_000,seasonal_single,[30.0],[30],1,1,1,12,0,0,True,True,period_detection_experiment_2\plots\seasonal_s...
1,seasonal_single_001,seasonal_single,[4.0],[4],1,1,1,12,0,0,True,True,period_detection_experiment_2\plots\seasonal_s...
2,seasonal_single_002,seasonal_single,[3.0],[3],1,1,1,12,0,0,True,True,period_detection_experiment_2\plots\seasonal_s...
3,seasonal_single_003,seasonal_single,[52.0],[52],1,1,1,12,0,0,True,True,period_detection_experiment_2\plots\seasonal_s...
4,seasonal_single_004,seasonal_single,[7.0],[7],1,1,1,12,0,0,True,True,period_detection_experiment_2\plots\seasonal_s...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
820,structural_break_trend_shift_020,structural_break_trend_shift,[],[],0,0,0,13,0,0,NaN,NaN,period_detection_experiment_2\plots\structural...
821,structural_break_trend_shift_021,structural_break_trend_shift,[],[],0,0,0,13,0,0,NaN,NaN,period_detection_experiment_2\plots\structural...
822,structural_break_trend_shift_022,structural_break_trend_shift,[],[],0,0,0,13,0,0,NaN,NaN,period_detection_experiment_2\plots\structural...
823,structural_break_trend_shift_023,structural_break_trend_shift,[],[],0,0,0,13,0,0,NaN,NaN,period_detection_experiment_2\plots\structural...


In [15]:
# Summary by example type: includes TP/TN/FP/FN counts, rates, and success_any/all rates.
summary_by_type

,example_type,n_series,true_period_count,detected_period_count,period_item_count,TP,TN,FP,FN,success_any_rate,success_all_rate,TP_rate,TN_rate,FP_rate,FN_rate
0,anomaly_collective_decay,25,0,0,325,0,325,0,0,NaN,NaN,NaN,1.000000,0.000000,NaN
1,anomaly_collective_gaussian,25,0,0,325,0,325,0,0,NaN,NaN,NaN,1.000000,0.000000,NaN
2,anomaly_collective_ramp,25,0,0,325,0,325,0,0,NaN,NaN,NaN,1.000000,0.000000,NaN
3,anomaly_collective_rectangular,25,0,0,325,0,325,0,0,NaN,NaN,NaN,1.000000,0.000000,NaN
4,anomaly_collective_triangular,25,0,0,325,0,325,0,0,NaN,NaN,NaN,1.000000,0.000000,NaN
5,anomaly_contextual_on_seasonal,25,25,20,325,20,300,0,5,0.8,0.8,0.80,1.000000,0.000000,0.20
6,anomaly_multiple_points,25,0,0,325,0,325,0,0,NaN,NaN,NaN,1.000000,0.000000,NaN
7,anomaly_single_point,25,0,0,325,0,325,0,0,NaN,NaN,NaN,1.000000,0.000000,NaN
8,deterministic_trend_cubic,25,0,0,325,0,325,0,0,NaN,NaN,NaN,1.000000,0.000000,NaN
9,deterministic_trend_damped,25,0,0,325,0,325,0,0,NaN,NaN,NaN,1.000000,0.000000,NaN


In [16]:
# Overall summary across all example types.
overall_summary

,example_type,n_series,true_period_count,detected_period_count,period_item_count,TP,TN,FP,FN,success_any_rate,success_all_rate,TP_rate,TN_rate,FP_rate,FN_rate
0,OVERALL,825,150,131,10725,127,10571,4,23,0.96,0.816,0.846667,0.999622,0.000378,0.153333


In [17]:
# Failed generations, if any. Usually inspect only if something was skipped.
failures_df

""


In [18]:

# =========================
# Problem cases only
# =========================

# False positives: at least one detected period that is not a true period.
false_positive_cases = series_view[series_view["FP"] > 0].copy()

# False negatives: at least one true period was missed.
false_negative_cases = series_view[series_view["FN"] > 0].copy()

false_positive_cases, false_negative_cases


(                      series_id            example_type true_periods  \
 141           stationary_ar_016           stationary_ar           []   
 185         stationary_arma_010         stationary_arma           []   
 222     stochastic_trend_rw_022     stochastic_trend_rw           []   
 307  stochastic_trend_arima_007  stochastic_trend_arima           []   
 
     detected_periods  true_period_count  detected_period_count  TP  TN  FP  \
 141             [13]                  0                      1   0  12   1   
 185             [63]                  0                      1   0  12   1   
 222             [63]                  0                      1   0  12   1   
 307             [63]                  0                      1   0  12   1   
 
      FN success_any success_all  \
 141   0         NaN         NaN   
 185   0         NaN         NaN   
 222   0         NaN         NaN   
 307   0         NaN         NaN   
 
                                              plot_pat

## How to inspect saved plots

The notebook does not display all plots. To inspect any example, open the path in `plot_path` from `series_view`, or use a single image display manually, for example:

```python
from IPython.display import Image, display
display(Image(filename=series_view.loc[0, "plot_path"]))
```

Do this only for a few selected examples, not inside a loop.


In [ ]:
# Optional: display one selected saved plot manually.
# from IPython.display import Image, display
# display(Image(filename=series_view.loc[0, "plot_path"]))

## Using the detector later on real data

For a real series, use `detect_periods_for_real_series`. You can provide the sampling frequency if known.

Examples:

```python
# Monthly real data
detected_periods, candidate_scores = detect_periods_for_real_series(
    real_series,
    sampling_frequency="monthly"
)

# Daily real data
detected_periods, candidate_scores = detect_periods_for_real_series(
    real_series,
    sampling_frequency="daily"
)

# Custom candidates
detected_periods, candidate_scores = detect_periods_for_real_series(
    real_series,
    custom_candidate_periods=[7, 14, 30, 365]
)
```

If `detected_periods == []`, do not apply seasonal extraction.
